# Bitcoin Transaction Data Exploration

本 notebook 对 `train.csv` 做基础探索，并结合 `test.csv` 与 `txs_edgelist.csv` 理解标签、时间和交易图结构。训练集较大，使用分块读取控制内存。

重点包括：数据质量、标签不平衡、时间分布、匿名特征统计、相关性、图节点度数，以及面向未来时间段的验证方式。

In [3]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
DATA_DIR = Path.cwd()
if not (DATA_DIR / 'train.csv').exists():
    DATA_DIR = Path('/Users/yichuan/Desktop/BT4012/Bitcoin')
TRAIN_PATH, TEST_PATH, EDGE_PATH = [DATA_DIR / name for name in ['train.csv', 'test.csv', 'txs_edgelist.csv']]
CHUNK_SIZE = 25_000
PLOT_SAMPLE_SIZE = 30_000
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style='whitegrid')
    PLOTTING_AVAILABLE = True
except ImportError as error:
    PLOTTING_AVAILABLE = False
    print(f'Optional plotting packages unavailable: {error}')

for path in [TRAIN_PATH, TEST_PATH, EDGE_PATH]:
    print(path.name, f'{path.stat().st_size / 1024**2:.1f} MB' if path.exists() else 'missing')

train.csv 439.6 MB
test.csv 47.3 MB
txs_edgelist.csv 4.3 MB


In [6]:
import os
from pathlib import Path

# Local data directory (no Google Drive needed)
DATA_DIR = Path('/Users/yichuan/Desktop/BT4012/Bitcoin')
TRAIN_PATH, TEST_PATH, EDGE_PATH = [DATA_DIR / name for name in ['train.csv', 'test.csv', 'txs_edgelist.csv']]

for path in [TRAIN_PATH, TEST_PATH, EDGE_PATH]:
    print(path.name, f'{path.stat().st_size / 1024**2:.1f} MB' if path.exists() else 'missing')


train.csv 439.6 MB
test.csv 47.3 MB
txs_edgelist.csv 4.3 MB


# 1 Schema and dataset sizes

In [ ]:
def preview(path, nrows=5):
    if not path.exists():
        print(f'File not found: {path}')
        return None
    frame = pd.read_csv(path, nrows=nrows)
    print(f'{path.name}: preview shape={frame.shape}')
    display(frame.head())
    display(frame.dtypes.to_frame('dtype').T)
    return frame

def count_rows(path):
    if not path.exists():
        return np.nan
    return sum(len(chunk) for chunk in pd.read_csv(path, usecols=[0], chunksize=CHUNK_SIZE))

train_preview = preview(TRAIN_PATH)
test_preview = preview(TEST_PATH)
edge_preview = preview(EDGE_PATH)
train_rows, test_rows, edge_rows = [count_rows(path) for path in [TRAIN_PATH, TEST_PATH, EDGE_PATH]]
print(f'rows: train={train_rows:,}, test={test_rows:,}, edges={edge_rows:,}')
feature_cols = [column for column in train_preview.columns if column.startswith('feat_')] if train_preview is not None else []
print(f'feature columns: {len(feature_cols)}')

#2 Full training-data quality checks


In [ ]:
def collect_quality(path):
    total = 0
    duplicates = 0
    ids = set()
    missing = None
    time_counts = {}
    label_counts = {}
    time_min, time_max = np.inf, -np.inf
    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE):
        total += len(chunk)
        duplicates += int(chunk.duplicated().sum())
        ids.update(chunk['txId'].dropna().astype(str))
        missing = chunk.isna().sum() if missing is None else missing.add(chunk.isna().sum(), fill_value=0)
        for key, value in chunk['time_step'].value_counts().items():
            time_counts[key] = time_counts.get(key, 0) + int(value)
        for key, value in chunk['label'].astype('string').fillna('<NA>').value_counts().items():
            label_counts[str(key)] = label_counts.get(str(key), 0) + int(value)
        time_min = min(time_min, chunk['time_step'].min())
        time_max = max(time_max, chunk['time_step'].max())
    return {'rows': total, 'unique_txId': len(ids), 'duplicates': duplicates,
            'missing': missing.sort_values(ascending=False),
            'time_counts': pd.Series(time_counts).sort_index(),
            'label_counts': pd.Series(label_counts).sort_index(),
            'time_min': time_min, 'time_max': time_max}

quality = collect_quality(TRAIN_PATH)
print(f"rows={quality['rows']:,}; unique txId={quality['unique_txId']:,}; duplicate rows within chunks={quality['duplicates']:,}")
print(f"time range={quality['time_min']} to {quality['time_max']}")
display(quality['label_counts'].rename('count').to_frame())
display(quality['missing'][quality['missing'] > 0].rename('missing_count').to_frame().head(20))

# 3 Labels, imbalance, time coverage, and feature distributions

In [ ]:
label_table = quality['label_counts'].rename('count').to_frame()
label_table['share'] = label_table['count'] / label_table['count'].sum()
display(label_table)
print('Observed labels:', sorted(label_table.index.astype(str)))
train_sample = pd.read_csv(TRAIN_PATH, nrows=PLOT_SAMPLE_SIZE)
test_sample = pd.read_csv(TEST_PATH, nrows=min(PLOT_SAMPLE_SIZE, int(test_rows)))
print('Feature summary:')
display(train_sample[feature_cols].describe().T.head(20))
display(train_sample[feature_cols].isna().mean().sort_values(ascending=False).head(20).rename('missing_rate').to_frame())

def all_times(path):
    return pd.concat((chunk['time_step'] for chunk in pd.read_csv(path, usecols=['time_step'], chunksize=CHUNK_SIZE)), ignore_index=True)

train_times, test_times = all_times(TRAIN_PATH), all_times(TEST_PATH)
coverage = pd.DataFrame({'dataset': ['train', 'test'], 'min_time_step': [train_times.min(), test_times.min()], 'max_time_step': [train_times.max(), test_times.max()], 'unique_time_steps': [train_times.nunique(), test_times.nunique()]})
display(coverage)
print('Test starts after train:', bool(test_times.min() > train_times.max()))
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    label_table['count'].plot.bar(ax=axes[0], title='Train label counts')
    quality['time_counts'].plot(ax=axes[1], marker='o', title='Train rows by time_step')
    plt.tight_layout(); plt.show()
    pd.DataFrame({'train': train_times.value_counts().sort_index(), 'test': test_times.value_counts().sort_index()}).fillna(0).plot.bar(figsize=(15, 5), title='Train/test time coverage')
    plt.tight_layout(); plt.show()
    train_sample[feature_cols[:20]].corr().pipe(sns.heatmap, cmap='vlag', center=0, square=True)
    plt.title('Correlation heatmap: first 20 features'); plt.tight_layout(); plt.show()
    train_sample[feature_cols[:6]].hist(figsize=(12, 8), bins=30); plt.tight_layout(); plt.show()
else:
    print('Plotting skipped because matplotlib/seaborn is unavailable.')

#4 Basic graph exploration


In [ ]:
edges = pd.read_csv(EDGE_PATH, usecols=['txId1', 'txId2'])
in_degree = edges['txId2'].value_counts().rename('in_degree')
out_degree = edges['txId1'].value_counts().rename('out_degree')
graph_features = pd.concat([in_degree, out_degree], axis=1).fillna(0)
graph_features['total_degree'] = graph_features['in_degree'] + graph_features['out_degree']
graph_features.index = graph_features.index.astype(str)
sample_ids = set(train_sample['txId'].astype(str))
matched = graph_features.loc[graph_features.index.intersection(sample_ids)]
print(f'edge rows={len(edges):,}; graph nodes={len(graph_features):,}; matched train sample={len(matched):,}/{len(sample_ids):,} ({len(matched) / len(sample_ids):.2%})')
display(graph_features.describe(percentiles=[.5, .9, .99]).T)
if PLOTTING_AVAILABLE:
    np.log1p(graph_features).plot.hist(bins=50, alpha=.75, figsize=(11, 5), title='Log-transformed degree distributions')
    plt.tight_layout(); plt.show()

# 5 Time-based validation and next steps

测试集代表未来时间段，推荐使用过去 `time_step` 训练、未来 `time_step` 验证，避免随机切分造成时间泄漏。

In [ ]:
available_times = np.sort(train_sample['time_step'].dropna().unique())
if len(available_times) >= 3:
    cutoff = available_times[int(len(available_times) * 0.8)]
    past = train_sample[train_sample.time_step < cutoff]
    future = train_sample[train_sample.time_step >= cutoff]
    print(f'cutoff={cutoff}; past rows={len(past):,}; future validation rows={len(future):,}')
    print(f'past range={past.time_step.min()}..{past.time_step.max()}; future range={future.time_step.min()}..{future.time_step.max()}')

print('Recommendations:')
print('- Exclude unknown/none/NA labels from supervised targets.')
print('- Evaluate PR-AUC, recall, precision, and calibrated probabilities for class imbalance.')
print('- Add graph degree features and inspect time distribution shift before modeling.')
print('- Use time-based validation and check for feature leakage before test predictions.')

#6 Supervised experiments and competition submissions

本节建立可复现实验流程。`train.csv` 中只有非空 `label` 的交易用于监督学习；未标注交易不当作负类，但可以参与图结构特征。`test.csv` 没有 `label`，只在最后用于生成提交概率。

比较两组模型：

- Model A: 匿名 `feat_*` 特征；
- Model B: `feat_*` 加上交易图的 `in_degree`、`out_degree` 和 `total_degree`。

验证采用时间切分，提交文件严格使用 `sample_submission.csv` 的 `index,target` 格式。

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score

# Load the complete tables. The edge-derived features use the observed full graph.
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
edges = pd.read_csv(EDGE_PATH, usecols=['txId1', 'txId2'])

feature_cols = [column for column in train.columns if column.startswith('feat_')]
base_cols = feature_cols

graph_features = pd.concat([
    edges['txId2'].value_counts().rename('in_degree'),
    edges['txId1'].value_counts().rename('out_degree'),
], axis=1).fillna(0)
graph_features.index = graph_features.index.astype(str)
graph_features['total_degree'] = graph_features['in_degree'] + graph_features['out_degree']

def add_graph_features(frame):
    result = frame.copy()
    graph_part = graph_features.reindex(result['txId'].astype(str)).fillna(0)
    for column in graph_features.columns:
        result[column] = graph_part[column].to_numpy()
    return result

train_model = add_graph_features(train)
test_model = add_graph_features(test)
labelled = train_model[train_model['label'].notna()].copy()
labelled['label'] = labelled['label'].astype(int)

train_part = labelled[labelled['time_step'] <= 28]
valid_part = labelled[labelled['time_step'] >= 29]
print(f'labelled={len(labelled):,}; train={len(train_part):,}; validation={len(valid_part):,}')
print(f'train positives={train_part.label.mean():.3f}; validation positives={valid_part.label.mean():.3f}')
print(f'graph nodes={len(graph_features):,}; edges={len(edges):,}')

In [ ]:
def class_weights(labels):
    counts = labels.value_counts()
    total = len(labels)
    return labels.map({int(cls): total / (2 * count) for cls, count in counts.items()}).to_numpy()

model_specs = {
    'baseline': feature_cols,
    'graph_degree': feature_cols + ['in_degree', 'out_degree', 'total_degree'],
}
validation_results = []
trained_models = {}

for model_name, columns in model_specs.items():
    model = HistGradientBoostingClassifier(
        max_iter=180,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42,
    )
    model.fit(
        train_part[columns],
        train_part['label'],
        sample_weight=class_weights(train_part['label']),
    )
    valid_probability = model.predict_proba(valid_part[columns])[:, 1]
    valid_prediction = (valid_probability >= 0.5).astype(int)
    validation_results.append({
        'model': model_name,
        'pr_auc': average_precision_score(valid_part['label'], valid_probability),
        'roc_auc': roc_auc_score(valid_part['label'], valid_probability),
        'precision_at_0.5': precision_score(valid_part['label'], valid_prediction, zero_division=0),
        'recall_at_0.5': recall_score(valid_part['label'], valid_prediction, zero_division=0),
    })
    trained_models[model_name] = model

validation_results = pd.DataFrame(validation_results).sort_values('pr_auc', ascending=False)
display(validation_results)

# Refit each experiment on every labelled training transaction before predicting test.
for model_name, columns in model_specs.items():
    final_model = HistGradientBoostingClassifier(
        max_iter=180,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42,
    )
    final_model.fit(
        labelled[columns],
        labelled['label'],
        sample_weight=class_weights(labelled['label']),
    )
    test_probability = final_model.predict_proba(test_model[columns])[:, 1]
    submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
    submission['target'] = test_probability
    assert submission['index'].equals(test['index'])
    assert submission['target'].between(0, 1).all()
    output_path = DATA_DIR / f'submission_{model_name}.csv'
    submission.to_csv(output_path, index=False)
    print(f'{output_path.name}: rows={len(submission):,}; target_mean={test_probability.mean():.4f}')

#7 Degree-feature ablation

上一轮结果只说明三种 degree 合在一起的效果。这里拆开 `in_degree`、`out_degree`，并测试 `log1p` 变换，观察哪一类图信号具有稳定价值。所有模型使用同一时间切分和同一分类器。

In [ ]:
ablation_train = train_model.copy()
ablation_test = test_model.copy()
for column in ['in_degree', 'out_degree', 'total_degree']:
    ablation_train[f'log_{column}'] = np.log1p(ablation_train[column])
    ablation_test[f'log_{column}'] = np.log1p(ablation_test[column])

def make_classifier():
    return HistGradientBoostingClassifier(
        max_iter=180,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42,
    )

def make_sample_weights(labels):
    counts = labels.value_counts()
    total = len(labels)
    weights = {int(cls): total / (2 * count) for cls, count in counts.items()}
    return labels.map(weights).to_numpy()

ablation_specs = {
    'in_degree_only': feature_cols + ['in_degree'],
    'out_degree_only': feature_cols + ['out_degree'],
    'log_degree_only': feature_cols + ['log_total_degree'],
    'log_in_out': feature_cols + ['log_in_degree', 'log_out_degree'],
}
ablation_results = []
for model_name, columns in ablation_specs.items():
    model = make_classifier()
    model.fit(
        ablation_train.loc[train_part.index, columns],
        train_part['label'],
        sample_weight=make_sample_weights(train_part['label']),
    )
    probability = model.predict_proba(ablation_train.loc[valid_part.index, columns])[:, 1]
    prediction = (probability >= 0.5).astype(int)
    ablation_results.append({
        'model': model_name,
        'pr_auc': average_precision_score(valid_part['label'], probability),
        'roc_auc': roc_auc_score(valid_part['label'], probability),
        'precision_at_0.5': precision_score(valid_part['label'], prediction, zero_division=0),
        'recall_at_0.5': recall_score(valid_part['label'], prediction, zero_division=0),
    })

ablation_results = pd.DataFrame(ablation_results).sort_values('pr_auc', ascending=False)
display(ablation_results)

for model_name, columns in ablation_specs.items():
    final_model = make_classifier()
    final_model.fit(
        ablation_train.loc[labelled.index, columns],
        labelled['label'],
        sample_weight=make_sample_weights(labelled['label']),
    )
    probability = final_model.predict_proba(ablation_test[columns])[:, 1]
    submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
    submission['target'] = probability
    assert submission['index'].equals(test['index'])
    assert submission['target'].between(0, 1).all()
    output_path = DATA_DIR / f'submission_{model_name}.csv'
    submission.to_csv(output_path, index=False)
    print(f'{output_path.name}: target_mean={probability.mean():.4f}')

#8 Neighbor feature aggregation

Degree 只描述连接数量。这里进一步计算少量高区分度匿名特征在前驱和后继邻居中的均值。邻居的 `label` 不会被使用，因此不会直接泄漏目标。为控制内存，只选择训练部分中单变量区分度最高的 12 个特征。

In [ ]:
from sklearn.feature_selection import f_classif

selection_X = train_part[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
selection_scores, _ = f_classif(selection_X, train_part['label'])
selected_neighbor_features = [
    feature_cols[index]
    for index in np.argsort(np.nan_to_num(selection_scores, nan=-np.inf))[-12:]
]
print('selected neighbor features:', selected_neighbor_features)

node_lookup = pd.concat([
    train_model[['txId'] + selected_neighbor_features],
    test_model[['txId'] + selected_neighbor_features],
], ignore_index=True)
node_lookup['txId'] = node_lookup['txId'].astype(str)
node_lookup = node_lookup.drop_duplicates('txId').set_index('txId')

edge_source_values = edges.copy()
edge_source_values['txId1'] = edge_source_values['txId1'].astype(str)
edge_source_values['txId2'] = edge_source_values['txId2'].astype(str)
source_values = edge_source_values[['txId1', 'txId2']].join(node_lookup, on='txId1')
pred_aggregates = source_values.groupby('txId2')[selected_neighbor_features].mean()
pred_aggregates.columns = [f'pred_mean_{column}' for column in selected_neighbor_features]

target_values = edge_source_values[['txId1', 'txId2']].join(node_lookup, on='txId2')
succ_aggregates = target_values.groupby('txId1')[selected_neighbor_features].mean()
succ_aggregates.columns = [f'succ_mean_{column}' for column in selected_neighbor_features]
neighbor_features = pred_aggregates.join(succ_aggregates, how='outer').fillna(0)

def add_neighbor_features(frame):
    result = frame.copy()
    mapped = neighbor_features.reindex(result['txId'].astype(str)).fillna(0)
    mapped.index = result.index
    return result.join(mapped)

neighbor_train = add_neighbor_features(train_model)
neighbor_test = add_neighbor_features(test_model)
neighbor_columns = list(neighbor_features.columns)
neighbor_specs = {
    'neighbor_means': feature_cols + neighbor_columns,
}
neighbor_results = []
for model_name, columns in neighbor_specs.items():
    model = make_classifier()
    model.fit(
        neighbor_train.loc[train_part.index, columns],
        train_part['label'],
        sample_weight=make_sample_weights(train_part['label']),
    )
    probability = model.predict_proba(neighbor_train.loc[valid_part.index, columns])[:, 1]
    prediction = (probability >= 0.5).astype(int)
    neighbor_results.append({
        'model': model_name,
        'pr_auc': average_precision_score(valid_part['label'], probability),
        'roc_auc': roc_auc_score(valid_part['label'], probability),
        'precision_at_0.5': precision_score(valid_part['label'], prediction, zero_division=0),
        'recall_at_0.5': recall_score(valid_part['label'], prediction, zero_division=0),
    })
display(pd.DataFrame(neighbor_results))

for model_name, columns in neighbor_specs.items():
    final_model = make_classifier()
    final_model.fit(
        neighbor_train.loc[labelled.index, columns],
        labelled['label'],
        sample_weight=make_sample_weights(labelled['label']),
    )
    probability = final_model.predict_proba(neighbor_test[columns])[:, 1]
    submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
    submission['target'] = probability
    assert submission['index'].equals(test['index'])
    output_path = DATA_DIR / f'submission_{model_name}.csv'
    submission.to_csv(output_path, index=False)
    print(f'{output_path.name}: target_mean={probability.mean():.4f}')

#9 Combined features: degree + neighbor aggregates

Previous experiments tested degree features and neighbor aggregates separately. Here we combine them and also enrich neighbor aggregates with **std, min, max** (not just mean), and use the top 30 features instead of 12.

In [ ]:
# --- Richer neighbor aggregation: top-30 features × {mean, std, min, max} ---

top_k = 30
selection_X = train_part[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
selection_scores, _ = f_classif(selection_X, train_part['label'])
top_k_features = [
    feature_cols[i]
    for i in np.argsort(np.nan_to_num(selection_scores, nan=-np.inf))[-top_k:]
]
print(f'Top-{top_k} features for neighbor aggregation: {top_k_features}')

# Build node lookup with top-k features (all train + test nodes)
node_lookup_k = pd.concat([
    train_model[['txId'] + top_k_features],
    test_model[['txId'] + top_k_features],
], ignore_index=True)
node_lookup_k['txId'] = node_lookup_k['txId'].astype(str)
node_lookup_k = node_lookup_k.drop_duplicates('txId').set_index('txId')

edges_str = edges.copy()
edges_str['txId1'] = edges_str['txId1'].astype(str)
edges_str['txId2'] = edges_str['txId2'].astype(str)

# Predecessor aggregation (edges point txId1 -> txId2, predecessors of v are txId1 where txId2==v)
source_vals = edges_str[['txId1', 'txId2']].join(node_lookup_k, on='txId1')
pred_agg = source_vals.groupby('txId2')[top_k_features].agg(['mean', 'std', 'min', 'max'])
pred_agg.columns = [f'pred_{stat}_{feat}' for feat, stat in pred_agg.columns]
pred_agg = pred_agg.fillna(0)

# Successor aggregation (successors of v are txId2 where txId1==v)
target_vals = edges_str[['txId1', 'txId2']].join(node_lookup_k, on='txId2')
succ_agg = target_vals.groupby('txId1')[top_k_features].agg(['mean', 'std', 'min', 'max'])
succ_agg.columns = [f'succ_{stat}_{feat}' for feat, stat in succ_agg.columns]
succ_agg = succ_agg.fillna(0)

rich_neighbor_features = pred_agg.join(succ_agg, how='outer').fillna(0)
rich_neighbor_cols = list(rich_neighbor_features.columns)
print(f'Rich neighbor feature columns: {len(rich_neighbor_cols)}')

def add_rich_neighbor(frame):
    result = frame.copy()
    mapped = rich_neighbor_features.reindex(result['txId'].astype(str)).fillna(0)
    mapped.index = result.index
    return result.join(mapped)

combined_train = add_rich_neighbor(ablation_train)
combined_test = add_rich_neighbor(ablation_test)

# Feature groups for combined experiments
degree_cols = ['in_degree', 'out_degree', 'total_degree',
               'log_in_degree', 'log_out_degree', 'log_total_degree']

combined_specs = {
    'feat_only':           feature_cols,
    'feat+degree':         feature_cols + degree_cols,
    'feat+rich_neighbor':  feature_cols + rich_neighbor_cols,
    'feat+degree+rich_neighbor': feature_cols + degree_cols + rich_neighbor_cols,
}

combined_results = []
for name, cols in combined_specs.items():
    model = make_classifier()
    model.fit(
        combined_train.loc[train_part.index, cols],
        train_part['label'],
        sample_weight=make_sample_weights(train_part['label']),
    )
    prob = model.predict_proba(combined_train.loc[valid_part.index, cols])[:, 1]
    pred = (prob >= 0.5).astype(int)
    combined_results.append({
        'model': name,
        'pr_auc': average_precision_score(valid_part['label'], prob),
        'roc_auc': roc_auc_score(valid_part['label'], prob),
        'precision': precision_score(valid_part['label'], pred, zero_division=0),
        'recall': recall_score(valid_part['label'], pred, zero_division=0),
    })

combined_results_df = pd.DataFrame(combined_results).sort_values('pr_auc', ascending=False)
display(combined_results_df)

# Generate submission for best combined model
best_combined_name = combined_results_df.iloc[0]['model']
best_combined_cols = combined_specs[best_combined_name]
final_model = make_classifier()
final_model.fit(
    combined_train.loc[labelled.index, best_combined_cols],
    labelled['label'],
    sample_weight=make_sample_weights(labelled['label']),
)
prob = final_model.predict_proba(combined_test[best_combined_cols])[:, 1]
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
submission['target'] = prob
output_path = DATA_DIR / f'submission_combined_best.csv'
submission.to_csv(output_path, index=False)
print(f'\nBest combined model: {best_combined_name}')
print(f'{output_path.name}: target_mean={prob.mean():.4f}')

#10 Rolling time-based validation

Replace the single train/valid split with 3-fold expanding-window validation. This better estimates generalization to future time steps and detects temporal overfitting.

In [ ]:
# --- Rolling time-based cross-validation ---

time_folds = [
    {'train': (1, 20), 'valid': (21, 25)},
    {'train': (1, 25), 'valid': (26, 30)},
    {'train': (1, 30), 'valid': (31, 35)},
]

# Use the best combined feature set from Section 9
best_cols = combined_specs[best_combined_name]

fold_results = []
for i, fold in enumerate(time_folds):
    t_lo, t_hi = fold['train']
    v_lo, v_hi = fold['valid']
    fold_train = combined_train[
        (combined_train['time_step'] >= t_lo) &
        (combined_train['time_step'] <= t_hi) &
        (combined_train['label'].notna())
    ].copy()
    fold_valid = combined_train[
        (combined_train['time_step'] >= v_lo) &
        (combined_train['time_step'] <= v_hi) &
        (combined_train['label'].notna())
    ].copy()
    if len(fold_valid) == 0:
        print(f'Fold {i}: no validation data in time_step {v_lo}-{v_hi}, skipping')
        continue
    fold_train['label'] = fold_train['label'].astype(int)
    fold_valid['label'] = fold_valid['label'].astype(int)

    model = make_classifier()
    model.fit(
        fold_train[best_cols],
        fold_train['label'],
        sample_weight=make_sample_weights(fold_train['label']),
    )
    prob = model.predict_proba(fold_valid[best_cols])[:, 1]
    pred = (prob >= 0.5).astype(int)
    fold_results.append({
        'fold': i,
        'train_time': f'{t_lo}-{t_hi}',
        'valid_time': f'{v_lo}-{v_hi}',
        'train_size': len(fold_train),
        'valid_size': len(fold_valid),
        'pos_rate_train': fold_train['label'].mean(),
        'pos_rate_valid': fold_valid['label'].mean(),
        'pr_auc': average_precision_score(fold_valid['label'], prob),
        'roc_auc': roc_auc_score(fold_valid['label'], prob),
        'precision': precision_score(fold_valid['label'], pred, zero_division=0),
        'recall': recall_score(fold_valid['label'], pred, zero_division=0),
    })

fold_df = pd.DataFrame(fold_results)
display(fold_df)
print(f'\nMean PR-AUC across folds: {fold_df["pr_auc"].mean():.4f} ± {fold_df["pr_auc"].std():.4f}')
print(f'Mean ROC-AUC across folds: {fold_df["roc_auc"].mean():.4f} ± {fold_df["roc_auc"].std():.4f}')

#11 Multi-model comparison: Logistic Regression, XGBoost, LightGBM

Compare multiple classifiers on the same feature set to find the best learner. Uses the single time split (train ≤ 28, valid ≥ 29) for fast iteration.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb

best_cols = combined_specs[best_combined_name]
pos_count = int(train_part['label'].sum())
neg_count = len(train_part) - pos_count
scale_pos = neg_count / max(pos_count, 1)

classifiers = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(
            class_weight='balanced', max_iter=1000, C=0.1, random_state=42
        )),
    ]),
    'HistGBT': make_classifier(),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.08,
        max_depth=6,
        scale_pos_weight=scale_pos,
        eval_metric='aucpr',
        tree_method='hist',
        random_state=42,
        verbosity=0,
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.08,
        num_leaves=31,
        scale_pos_weight=scale_pos,
        random_state=42,
        verbose=-1,
    ),
}

multi_model_results = []
trained_multi = {}

for name, clf in classifiers.items():
    use_weight = name == 'HistGBT'
    fit_kwargs = {}
    if use_weight:
        fit_kwargs['sample_weight'] = make_sample_weights(train_part['label'])

    clf.fit(combined_train.loc[train_part.index, best_cols], train_part['label'], **fit_kwargs)
    prob = clf.predict_proba(combined_train.loc[valid_part.index, best_cols])[:, 1]
    pred = (prob >= 0.5).astype(int)
    multi_model_results.append({
        'model': name,
        'pr_auc': average_precision_score(valid_part['label'], prob),
        'roc_auc': roc_auc_score(valid_part['label'], prob),
        'precision': precision_score(valid_part['label'], pred, zero_division=0),
        'recall': recall_score(valid_part['label'], pred, zero_division=0),
    })
    trained_multi[name] = clf

multi_model_df = pd.DataFrame(multi_model_results).sort_values('pr_auc', ascending=False)
display(multi_model_df)

#12 Ensemble: blending top models

Average predicted probabilities from the top tree-based models. Blending diverse learners typically improves calibration and reduces variance.

In [ ]:
# --- Ensemble: average of HistGBT, XGBoost, LightGBM ---

ensemble_models = ['HistGBT', 'XGBoost', 'LightGBM']
valid_probs = {}
for name in ensemble_models:
    valid_probs[name] = trained_multi[name].predict_proba(
        combined_train.loc[valid_part.index, best_cols]
    )[:, 1]

# Simple average ensemble
ensemble_prob = np.mean([valid_probs[n] for n in ensemble_models], axis=0)
ensemble_pred = (ensemble_prob >= 0.5).astype(int)

ensemble_result = {
    'model': 'Ensemble (avg)',
    'pr_auc': average_precision_score(valid_part['label'], ensemble_prob),
    'roc_auc': roc_auc_score(valid_part['label'], ensemble_prob),
    'precision': precision_score(valid_part['label'], ensemble_pred, zero_division=0),
    'recall': recall_score(valid_part['label'], ensemble_pred, zero_division=0),
}

# Combine with individual results
all_results = pd.concat([multi_model_df, pd.DataFrame([ensemble_result])], ignore_index=True)
all_results = all_results.sort_values('pr_auc', ascending=False)
display(all_results)

# --- Generate ensemble submission ---
# Refit all 3 models on full labelled data, then average test predictions
ensemble_test_probs = []
for name, clf_template in classifiers.items():
    if name not in ensemble_models:
        continue
    import copy
    clf = copy.deepcopy(clf_template) if not isinstance(clf_template, Pipeline) else clf_template
    use_weight = name == 'HistGBT'
    fit_kwargs = {}
    if use_weight:
        fit_kwargs['sample_weight'] = make_sample_weights(labelled['label'])
    clf.fit(combined_train.loc[labelled.index, best_cols], labelled['label'], **fit_kwargs)
    ensemble_test_probs.append(
        clf.predict_proba(combined_test[best_cols])[:, 1]
    )

ensemble_test_prob = np.mean(ensemble_test_probs, axis=0)
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
submission['target'] = ensemble_test_prob
output_path = DATA_DIR / 'submission_ensemble.csv'
submission.to_csv(output_path, index=False)
print(f'{output_path.name}: target_mean={ensemble_test_prob.mean():.4f}')

#13 Higher-order graph features: 2-hop aggregates, PageRank, clustering coefficient, component size

Degree and 1-hop neighbor means only capture immediate connectivity. This section adds:
- **2-hop neighbor aggregates** — statistics of neighbors-of-neighbors
- **PageRank** — global importance / flow centrality
- **Local clustering coefficient** — how tightly connected a node's neighborhood is (fraud rings form clusters)
- **Connected component size** — isolates vs. large components

In [ ]:
import networkx as nx
from collections import defaultdict
import lightgbm as lgb

# Build directed graph (reuse edges already loaded)
G = nx.from_pandas_edgelist(edges, source='txId1', target='txId2', create_using=nx.DiGraph())
G_undirected = G.to_undirected()
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# --- PageRank ---
pagerank = nx.pagerank(G, alpha=0.85, max_iter=100)
print(f'PageRank computed for {len(pagerank)} nodes')

# --- HITS ---
hubs, authorities = nx.hits(G, max_iter=100, normalized=True)

# --- Local clustering coefficient (undirected) ---
clustering = nx.clustering(G_undirected)

# --- Connected component size (undirected) ---
component_size = {}
for comp in nx.connected_components(G_undirected):
    sz = len(comp)
    for node in comp:
        component_size[node] = sz

# --- 2-hop neighbor aggregate (mean of 1-hop neighbors' degree) ---
# For each node, collect the set of 2-hop neighbors via undirected graph
two_hop_in_degree = {}
two_hop_out_degree = {}
two_hop_count = {}
for node in G.nodes():
    neighbors_1 = set(G_undirected.neighbors(node))
    neighbors_2 = set()
    for n1 in neighbors_1:
        neighbors_2.update(G_undirected.neighbors(n1))
    neighbors_2.discard(node)
    neighbors_2 -= neighbors_1
    two_hop_count[node] = len(neighbors_2)
    if neighbors_2:
        two_hop_in_degree[node] = np.mean([G.in_degree(n) for n in neighbors_2])
        two_hop_out_degree[node] = np.mean([G.out_degree(n) for n in neighbors_2])
    else:
        two_hop_in_degree[node] = 0.0
        two_hop_out_degree[node] = 0.0

# Assemble into DataFrame
graph_advanced = pd.DataFrame({
    'pagerank': pagerank,
    'hub_score': hubs,
    'authority_score': authorities,
    'clustering_coeff': clustering,
    'component_size': component_size,
    'two_hop_count': two_hop_count,
    'two_hop_mean_in_deg': two_hop_in_degree,
    'two_hop_mean_out_deg': two_hop_out_degree,
})
graph_advanced.index = graph_advanced.index.astype(str)
graph_advanced['log_pagerank'] = np.log1p(graph_advanced['pagerank'])
graph_advanced['log_component_size'] = np.log1p(graph_advanced['component_size'])
advanced_graph_cols = list(graph_advanced.columns)
print(f'Advanced graph features: {advanced_graph_cols}')

def add_advanced_graph(frame):
    result = frame.copy()
    mapped = graph_advanced.reindex(result['txId'].astype(str)).fillna(0)
    mapped.index = result.index
    for col in advanced_graph_cols:
        result[col] = mapped[col].values
    return result

combined_train = add_advanced_graph(combined_train)
combined_test = add_advanced_graph(combined_test)

# Evaluate
graph_adv_specs = {
    'best_cols (baseline)': best_cols,
    'best+advanced_graph': best_cols + advanced_graph_cols,
}
graph_adv_results = []
for name, cols in graph_adv_specs.items():
    clf = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.08, num_leaves=31,
        scale_pos_weight=scale_pos, random_state=42, verbose=-1,
    )
    clf.fit(combined_train.loc[train_part.index, cols], train_part['label'])
    prob = clf.predict_proba(combined_train.loc[valid_part.index, cols])[:, 1]
    pred = (prob >= 0.5).astype(int)
    graph_adv_results.append({
        'model': name,
        'pr_auc': average_precision_score(valid_part['label'], prob),
        'roc_auc': roc_auc_score(valid_part['label'], prob),
        'precision': precision_score(valid_part['label'], pred, zero_division=0),
        'recall': recall_score(valid_part['label'], pred, zero_division=0),
    })

graph_adv_df = pd.DataFrame(graph_adv_results).sort_values('pr_auc', ascending=False)
display(graph_adv_df)

#14 Personalized PageRank (PPR) for Fraud Detection

Standard PageRank acts as a global centrality measure. **Personalized PageRank (PPR)** biases the random walks to start/teleport back to a specific subset of nodes—the known fraudulent (illicit) seed transactions.

Nodes with high PPR scores under this personalization vector are structurally close to multiple known fraud nodes, making PPR an extremely potent feature for illicit transaction detection.

In [ ]:
# --- Personalized PageRank Calculation ---
import networkx as nx

# Prepare the personalization vector
# We bias the teleportation vector toward known fraudulent transactions in the training set
fraud_seeds = labelled[labelled['label'] == 1]['txId'].astype(str).tolist()

# Initialize with uniform distribution over all nodes
personalization = {node: 0.0 for node in G.nodes()}

# Distribute personalization weight equally among the known fraud seed nodes
valid_seeds = [seed for seed in fraud_seeds if seed in personalization]
if valid_seeds:
    weight = 1.0 / len(valid_seeds)
    for seed in valid_seeds:
        personalization[seed] = weight

    print(f"Computing Personalized PageRank with {len(valid_seeds)} fraud seed nodes...")
    # Compute PPR (typically alpha is set between 0.85 and 0.90)
    ppr = nx.pagerank(G, alpha=0.85, personalization=personalization, max_iter=100)
else:
    print("No fraud seeds found in the graph. Falling back to standard PageRank.")
    ppr = pagerank

# Map PPR scores to train and test sets
combined_train['personalized_pagerank'] = combined_train['txId'].astype(str).map(ppr).fillna(0.0)
combined_test['personalized_pagerank'] = combined_test['txId'].astype(str).map(ppr).fillna(0.0)

# Evaluate Personalized PageRank against our baseline
ppr_specs = {
    'best_cols (baseline)': best_cols,
    'best+personalized_pagerank': best_cols + ['personalized_pagerank'],
}

ppr_results = []
for name, cols in ppr_specs.items():
    clf = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.08, num_leaves=31,
        scale_pos_weight=scale_pos, random_state=42, verbose=-1,
    )
    clf.fit(combined_train.loc[train_part.index, cols], train_part['label'])
    prob = clf.predict_proba(combined_train.loc[valid_part.index, cols])[:, 1]
    pred = (prob >= 0.5).astype(int)
    ppr_results.append({
        'model': name,
        'pr_auc': average_precision_score(valid_part['label'], prob),
        'roc_auc': roc_auc_score(valid_part['label'], prob),
        'precision': precision_score(valid_part['label'], pred, zero_division=0),
        'recall': recall_score(valid_part['label'], pred, zero_division=0),
    })

ppr_df = pd.DataFrame(ppr_results).sort_values('pr_auc', ascending=False)
display(ppr_df)

In [ ]:
# --- Refit best+personalized_pagerank model on full labelled data and generate submission ---

ppr_cols = best_cols + ['personalized_pagerank']

final_ppr_model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.08,
    num_leaves=31,
    scale_pos_weight=scale_pos,
    random_state=42,
    verbose=-1,
)

# Train on ALL labelled training data
final_ppr_model.fit(
    combined_train.loc[labelled.index, ppr_cols],
    labelled['label']
)

# Predict on test data
test_prob_ppr = final_ppr_model.predict_proba(combined_test[ppr_cols])[:, 1]

# Prepare submission
sub_ppr = pd.read_csv(DATA_DIR / 'sample_submission.csv')
sub_ppr['target'] = test_prob_ppr

output_file = DATA_DIR / 'submission_personalized_pagerank.csv'
sub_ppr.to_csv(output_file, index=False)

print(f"成功生成 CSV 文件: {output_file.name}")
print(f"预测均值 (target_mean): {test_prob_ppr.mean():.4f}")
print(f"预测标准差 (target_std): {test_prob_ppr.std():.4f}")


#15 Corrected Personalized PageRank (PPR)

We fix the data type mismatch by mapping the fraud seeds' node IDs to integers (`int`), ensuring they align with the node types in the NetworkX graph `G`.

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score

# Prepare the personalization vector
# Convert seeds to integers to match the node representation in G
fraud_seeds_int = labelled[labelled['label'] == 1]['txId'].astype(int).tolist()

# Initialize with 0.0 for all nodes
personalization = {node: 0.0 for node in G.nodes()}

# Distribute personalization weight equally among the known fraud seed nodes
valid_seeds_int = [seed for seed in fraud_seeds_int if seed in personalization]

if valid_seeds_int:
    weight = 1.0 / len(valid_seeds_int)
    for seed in valid_seeds_int:
        personalization[seed] = weight
    print(f"Success! Computing Corrected Personalized PageRank with {len(valid_seeds_int)} fraud seed nodes...")
    corrected_ppr = nx.pagerank(G, alpha=0.85, personalization=personalization, max_iter=100)
else:
    print("Warning: Still no fraud seeds found. Falling back to standard PageRank.")
    corrected_ppr = pagerank

# Map PPR scores back to train and test sets using float node conversion for lookup
combined_train['corrected_personalized_pagerank'] = combined_train['txId'].astype(int).map(corrected_ppr).fillna(0.0)
combined_test['corrected_personalized_pagerank'] = combined_test['txId'].astype(int).map(corrected_ppr).fillna(0.0)

# Evaluate the model with corrected PPR
corrected_ppr_cols = best_cols + ['corrected_personalized_pagerank']

clf_ppr = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.08,
    num_leaves=31,
    scale_pos_weight=scale_pos,
    random_state=42,
    verbose=-1
)

clf_ppr.fit(combined_train.loc[train_part.index, corrected_ppr_cols], train_part['label'])
prob_ppr = clf_ppr.predict_proba(combined_train.loc[valid_part.index, corrected_ppr_cols])[:, 1]
pred_ppr = (prob_ppr >= 0.5).astype(int)

corrected_ppr_result = {
    'model': 'best+corrected_personalized_pagerank',
    'pr_auc': average_precision_score(valid_part['label'], prob_ppr),
    'roc_auc': roc_auc_score(valid_part['label'], prob_ppr),
    'precision': precision_score(valid_part['label'], pred_ppr, zero_division=0),
    'recall': recall_score(valid_part['label'], pred_ppr, zero_division=0)
}

display(pd.DataFrame([corrected_ppr_result]))

In [ ]:
# Refit on ALL labelled training data and generate a new submission
final_corrected_ppr_model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.08,
    num_leaves=31,
    scale_pos_weight=scale_pos,
    random_state=42,
    verbose=-1
)

final_corrected_ppr_model.fit(
    combined_train.loc[labelled.index, corrected_ppr_cols],
    labelled['label']
)

test_prob_corrected_ppr = final_corrected_ppr_model.predict_proba(combined_test[corrected_ppr_cols])[:, 1]

sub_corrected_ppr = pd.read_csv(DATA_DIR / 'sample_submission.csv')
sub_corrected_ppr['target'] = test_prob_corrected_ppr
output_file_ppr = DATA_DIR / 'submission_corrected_personalized_pagerank.csv'
sub_corrected_ppr.to_csv(output_file_ppr, index=False)

print(f"成功生成 CSV 文件: {output_file_ppr.name}")
print(f"预测均值 (target_mean): {test_prob_corrected_ppr.mean():.4f}")

Section 27d — Leakage-Free Personalized PageRank

In [ ]:
# ============================================================
# Section 27d — Leakage-Free Personalized PageRank
# ============================================================

import numpy as np
import pandas as pd
import networkx as nx
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score
)

# ------------------------------------------------------------
# 1. PPR calculation function
# ------------------------------------------------------------

def compute_ppr(G, seed_ids, alpha=0.85):

    graph_nodes = set(G.nodes())

    valid_seeds = [
        int(node)
        for node in seed_ids
        if int(node) in graph_nodes
    ]

    if not valid_seeds:
        raise ValueError("No valid fraud seeds found in graph!")

    personalization = {
        node: 0.0
        for node in G.nodes()
    }

    weight = 1.0 / len(valid_seeds)

    for node in valid_seeds:
        personalization[node] = weight

    return nx.pagerank(
        G,
        alpha=alpha,
        personalization=personalization,
        max_iter=200,
        tol=1e-6
    )


# ------------------------------------------------------------
# 2. Initialize feature
# ------------------------------------------------------------

combined_train['ppr_oof'] = np.nan
combined_test['ppr_oof'] = np.nan

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

X_train = train_part.copy()
y_train = train_part['label']

print("Generating OOF PPR features...")


# ------------------------------------------------------------
# 3. Generate OOF PPR for training nodes
# ------------------------------------------------------------

for fold, (fit_idx, hold_idx) in enumerate(
    skf.split(X_train, y_train)
):

    fold_train = X_train.iloc[fit_idx]
    fold_hold = X_train.iloc[hold_idx]

    fraud_seeds = (
        fold_train.loc[
            fold_train['label'] == 1,
            'txId'
        ]
        .astype(int)
        .tolist()
    )

    print(
        f"Fold {fold + 1}: "
        f"{len(fraud_seeds)} fraud seeds"
    )

    fold_ppr = compute_ppr(
        G,
        fraud_seeds,
        alpha=0.85
    )

    combined_train.loc[
        fold_hold.index,
        'ppr_oof'
    ] = (
        fold_hold['txId']
        .astype(int)
        .map(fold_ppr)
        .fillna(0.0)
        .to_numpy()
    )


# ------------------------------------------------------------
# 4. Validation PPR
# ------------------------------------------------------------

full_train_seeds = (
    train_part.loc[
        train_part['label'] == 1,
        'txId'
    ]
    .astype(int)
    .tolist()
)

full_train_ppr = compute_ppr(
    G,
    full_train_seeds,
    alpha=0.85
)

combined_train.loc[
    valid_part.index,
    'ppr_oof'
] = (
    valid_part['txId']
    .astype(int)
    .map(full_train_ppr)
    .fillna(0.0)
    .to_numpy()
)


# ------------------------------------------------------------
# 5. Test PPR
# ------------------------------------------------------------

# For final test inference, all available labelled training
# nodes may be used as seeds, provided competition rules allow.

all_train_seeds = (
    labelled.loc[
        labelled['label'] == 1,
        'txId'
    ]
    .astype(int)
    .tolist()
)

test_ppr = compute_ppr(
    G,
    all_train_seeds,
    alpha=0.85
)

combined_test['ppr_oof'] = (
    combined_test['txId']
    .astype(int)
    .map(test_ppr)
    .fillna(0.0)
)


# ------------------------------------------------------------
# 6. Compare baseline vs PPR
# ------------------------------------------------------------

pos_count = int(train_part['label'].sum())
neg_count = len(train_part) - pos_count

scale_pos = neg_count / max(pos_count, 1)

experiments = {
    'baseline': best_cols,
    'baseline + OOF PPR': best_cols + ['ppr_oof']
}

results = []

for name, cols in experiments.items():

    model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.08,
        num_leaves=31,
        scale_pos_weight=scale_pos,
        random_state=42,
        verbose=-1
    )

    model.fit(
        combined_train.loc[train_part.index, cols],
        train_part['label']
    )

    prob = model.predict_proba(
        combined_train.loc[valid_part.index, cols]
    )[:, 1]

    pred = (prob >= 0.5).astype(int)

    results.append({
        'model': name,
        'pr_auc': average_precision_score(
            valid_part['label'], prob
        ),
        'roc_auc': roc_auc_score(
            valid_part['label'], prob
        ),
        'precision': precision_score(
            valid_part['label'], pred,
            zero_division=0
        ),
        'recall': recall_score(
            valid_part['label'], pred,
            zero_division=0
        )
    })

results_df = pd.DataFrame(results)

display(results_df)

##15b Personalized PageRank Feature Optimization

Personalized PageRank (PPR) scores are mathematically guaranteed to sum to 1 across all 200,000+ nodes in our graph, resulting in extremely tiny values (e.g., $10^{-6}$ to $10^{-8}$) with highly skewed distributions. Tree-based models like LightGBM often struggle to split effectively on such raw, unscaled floating-point centralities.

To make this extremely powerful global graph signal fully digestible by LightGBM, we apply two optimizations:
1. **Logarithmic Scaling (`log1p(ppr * scale)`)**: Magnifies the subtle score variations among nodes.
2. **Rank Scaling (Percentile transformation)**: Standardizes the feature into a uniform scale $[0, 1]$, neutralizing potential extreme outliers.

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score

# 1. 检查当前 corrected_personalized_pagerank 的基础分布
raw_ppr_train = combined_train['corrected_personalized_pagerank'].values
print(f"原始 PPR 统计信息 (Train): Mean={raw_ppr_train.mean():.2e}, Max={raw_ppr_train.max():.2e}, Min={raw_ppr_train.min():.2e}")

# 2. 特征变换一：Log1p 变换 (乘以一个缩放因子放大特征范围)
scale_factor = 1e6
combined_train['ppr_log'] = np.log1p(combined_train['corrected_personalized_pagerank'] * scale_factor)
combined_test['ppr_log'] = np.log1p(combined_test['corrected_personalized_pagerank'] * scale_factor)

# 3. 特征变换二：Rank 变换 (转换到 [0, 1] 均匀空间)
# 使用 rankdata 计算所有节点在合并集上的百分比排名，避免训练集和测试集分布割裂
all_ppr = pd.concat([combined_train['corrected_personalized_pagerank'], combined_test['corrected_personalized_pagerank']], ignore_index=True)
all_ppr_ranks = rankdata(all_ppr) / len(all_ppr)

combined_train['ppr_rank'] = all_ppr_ranks[:len(combined_train)]
combined_test['ppr_rank'] = all_ppr_ranks[len(combined_train):]

print("\n变换后特征示例:")
display(combined_train[['corrected_personalized_pagerank', 'ppr_log', 'ppr_rank']].describe().T)

In [ ]:
# 4. 将优化后的特征代入验证流程进行评估
oppr_specs = {
    'best_cols (baseline)': best_cols,
    'best+raw_ppr': best_cols + ['corrected_personalized_pagerank'],
    'best+ppr_log': best_cols + ['ppr_log'],
    'best+ppr_rank': best_cols + ['ppr_rank'],
    'best+ppr_all_transforms': best_cols + ['corrected_personalized_pagerank', 'ppr_log', 'ppr_rank']
}

oppr_results = []
for name, cols in oppr_specs.items():
    clf = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.08,
        num_leaves=31,
        scale_pos_weight=scale_pos,
        random_state=42,
        verbose=-1
    )
    clf.fit(combined_train.loc[train_part.index, cols], train_part['label'])
    prob = clf.predict_proba(combined_train.loc[valid_part.index, cols])[:, 1]
    pred = (prob >= 0.5).astype(int)
    oppr_results.append({
        'model': name,
        'pr_auc': average_precision_score(valid_part['label'], prob),
        'roc_auc': roc_auc_score(valid_part['label'], prob),
        'precision': precision_score(valid_part['label'], pred, zero_division=0),
        'recall': recall_score(valid_part['label'], pred, zero_division=0)
    })

display(pd.DataFrame(oppr_results).sort_values('pr_auc', ascending=False))

In [ ]:
# 5. 重新拟合表现最佳的特征配置并生成新的单模型提交文件
best_ppr_spec = 'best+ppr_all_transforms'
final_cols = oppr_specs[best_ppr_spec]

final_model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.08,
    num_leaves=31,
    scale_pos_weight=scale_pos,
    random_state=42,
    verbose=-1
)

final_model.fit(combined_train.loc[labelled.index, final_cols], labelled['label'])
test_prob_opt_ppr = final_model.predict_proba(combined_test[final_cols])[:, 1]

sub_opt_ppr = pd.read_csv(DATA_DIR / 'sample_submission.csv')
sub_opt_ppr['target'] = test_prob_opt_ppr
output_file_opt_ppr = DATA_DIR / 'submission_optimized_personalized_pagerank.csv'
sub_opt_ppr.to_csv(output_file_opt_ppr, index=False)

print(f"\n优化后 PPR 单模型提交生成成功: {output_file_opt_ppr.name}")
print(f"预测分布均值 (target_mean): {test_prob_opt_ppr.mean():.4f}")
print(f"预测分布标准差 (target_std): {test_prob_opt_ppr.std():.4f}")

##15c. Rank-Averaging Ensemble for Robust Leaderboard Scores

为了克服图上全局传播（如 Personalized PageRank）在未来时间步长（Out-of-Time test set）上的泛化性退化问题，我们选择回到局部安全邻居特征（1-hop neighbor features, temporal features, anomalies），并对我们已有的几组不含未来图结构泄漏的最强模型预测结果进行 **Rank-Averaging (秩平均) 融合**。

Rank-Averaging 的公式为：
$$RankAvg(x) = \frac{1}{M} \sum_{m=1}^{M} Rank(P_m(x))$$
其中 $Rank(P_m(x))$ 将概率转化为区间在 $[0, 1]$ 之间的百分比排名。这能有效消除多模型融合时各分类器概率校准尺度（Scale）不一致的影响，是表格与图混合任务中最稳健的提分策略。

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata

# 定义参与集成的优秀、无泄露提交文件
ensemble_files = [
    'submission_final_ensemble.csv',
    'submission_combined_best.csv',
    'submission_tuned_lgbm.csv'
]

# 确保文件存在
valid_files = [DATA_DIR / f for f in ensemble_files if (DATA_DIR / f).exists()]
print(f"参与集成的有效文件: {[f.name for f in valid_files]}")

if len(valid_files) >= 2:
    preds_df = pd.DataFrame()

    for i, file_path in enumerate(valid_files):
        df = pd.read_csv(file_path)
        if i == 0:
            preds_df['index'] = df['index']

        # 将预测值转化为百分比 rank [0, 1]
        preds_df[f'rank_pred_{i}'] = rankdata(df['target']) / len(df)

    # 计算 Rank 均值
    rank_cols = [c for c in preds_df.columns if c.startswith('rank_pred_')]
    preds_df['target'] = preds_df[rank_cols].mean(axis=1)

    # 生成最终 Rank 提交
    sub_rank = preds_df[['index', 'target']]
    output_file_rank = DATA_DIR / 'submission_rank_average_ensemble.csv'
    sub_rank.to_csv(output_file_rank, index=False)

    print(f"\n成功生成 Rank-Averaging 集成文件: {output_file_rank.name}")
    print(f"集成后提交分布均值: {preds_df['target'].mean():.4f}")
    print(f"集成后提交分布标准差: {preds_df['target'].std():.4f}")
else:
    print("错误: 未能找到足够的基础提交文件进行 Rank-Averaging 融合，请确保它们已正确保存在 DATA_DIR 下。")

##15d Robust & Natural Probability Ensemble via Rank-to-Gauss (Probit) Transform

为了解决普通 Rank-Averaging 导致的输出呈均匀分布（丢失了自然概率分布形态与物理意义）的问题，我们引入高斯投影变换。该方法先将各模型的预测值转换为分位数排名，再通过高斯的逆累积分布函数（`scipy.stats.norm.ppf`）将它们投影到正态分布空间。在此空间进行融合均值后，再映射回平滑的自然概率区间。

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata, norm

# 定义基础提交文件
ensemble_files = [
    'submission_final_ensemble.csv',
    'submission_combined_best.csv',
    'submission_tuned_lgbm.csv'
]

valid_files = [DATA_DIR / f for f in ensemble_files if (DATA_DIR / f).exists()]
print(f"参与自然融合的有效文件: {[f.name for f in valid_files]}")

if len(valid_files) >= 2:
    latent_scores = []

    for file_path in valid_files:
        df = pd.read_csv(file_path)
        # 1. 转换为百分比 rank，并微调边界防止 norm.ppf 产生 inf
        ranks = rankdata(df['target']) / (len(df) + 1)
        # 2. 映射到高斯潜空间
        z_scores = norm.ppf(ranks)
        latent_scores.append(z_scores)

    # 3. 在高斯潜空间求均值
    mean_latent = np.mean(latent_scores, axis=0)

    # 4. 使用 Sigmoid 恢复到具有自然偏态的概率空间 [0, 1]
    # 使用标准差缩放让输出概率尺度更符合原本分类器的输出均值
    natural_probs = 1 / (1 + np.exp(-mean_latent * 0.8))

    # 生成最终提交
    sub_natural = pd.DataFrame({
        'index': pd.read_csv(valid_files[0])['index'],
        'target': natural_probs
    })

    output_file_natural = DATA_DIR / 'submission_natural_blend.csv'
    sub_natural.to_csv(output_file_natural, index=False)

    print(f"\n成功生成自然概率融合提交文件: {output_file_natural.name}")
    print(f"融合后均值 (target_mean): {sub_natural['target'].mean():.4f}")
    print(f"融合后标准差 (target_std): {sub_natural['target'].std():.4f}")
    print(f"前 5 行预测结果预览:\n", sub_natural.head())
else:
    print("错误: 基础文件不足。")

In [ ]:
print(f'总特征数: {len(combined_train.columns)}')
print('\n前 30 个特征名示例:')
print(list(combined_train.columns[:30]))

print('\n高级图结构特征及新引入的特征:')
advanced_cols = [c for c in combined_train.columns if 'pagerank' in c or 'degree' in c or 'coeff' in c or 'component' in c or 'score' in c]
print(advanced_cols)

#16

In [ ]:
# ============================================================
# Section 29 — Directional 2-Hop Neighbor Aggregation
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb

# ------------------------------------------------------------
# 1. Reuse top-30 selected original features
# ------------------------------------------------------------

base_features = top_k_features.copy()

node_features = node_lookup_k[base_features].copy()

print("Original features:", len(base_features))
print("Available nodes:", len(node_features))


# ------------------------------------------------------------
# 2. Helper: directional neighbor mean aggregation
# ------------------------------------------------------------

def aggregate_neighbor_mean(
    node_values,
    edges,
    direction
):

    values = node_values.copy()
    values.index = values.index.astype(str)

    if direction == "pred":

        joined = edges[
            ['txId1', 'txId2']
        ].join(
            values,
            on='txId1'
        )

        aggregated = joined.groupby(
            'txId2'
        )[values.columns].mean()

    elif direction == "succ":

        joined = edges[
            ['txId1', 'txId2']
        ].join(
            values,
            on='txId2'
        )

        aggregated = joined.groupby(
            'txId1'
        )[values.columns].mean()

    else:
        raise ValueError(
            "direction must be pred or succ"
        )

    return aggregated


# ------------------------------------------------------------
# 3. First-hop mean features
# ------------------------------------------------------------

first_pred = aggregate_neighbor_mean(
    node_features,
    edges_str,
    "pred"
)

first_succ = aggregate_neighbor_mean(
    node_features,
    edges_str,
    "succ"
)

# Align all nodes
first_pred = first_pred.reindex(
    node_features.index
).fillna(0)

first_succ = first_succ.reindex(
    node_features.index
).fillna(0)


# ------------------------------------------------------------
# 4. Second-hop directional combinations
# ------------------------------------------------------------

second_hop = {}

for first_direction, first_values in [
    ("pred", first_pred),
    ("succ", first_succ)
]:

    for second_direction in [
        "pred",
        "succ"
    ]:

        aggregated = aggregate_neighbor_mean(
            first_values,
            edges_str,
            second_direction
        )

        aggregated = aggregated.reindex(
            node_features.index
        ).fillna(0)

        prefix = (
            f"{second_direction}_"
            f"{first_direction}_mean_"
        )

        aggregated.columns = [
            prefix + col
            for col in aggregated.columns
        ]

        second_hop[
            prefix
        ] = aggregated


# ------------------------------------------------------------
# 5. Combine 2-hop features
# ------------------------------------------------------------

two_hop_features = pd.concat(
    second_hop.values(),
    axis=1
)

two_hop_cols = list(
    two_hop_features.columns
)

print(
    "Generated 2-hop features:",
    len(two_hop_cols)
)


# ------------------------------------------------------------
# 6. Add features to train and test
# ------------------------------------------------------------

def add_two_hop_features(frame):

    result = frame.copy()

    mapped = two_hop_features.reindex(
        result['txId'].astype(str)
    ).fillna(0)

    mapped.index = result.index

    return result.join(mapped)


combined_train_2hop = add_two_hop_features(
    combined_train
)

combined_test_2hop = add_two_hop_features(
    combined_test
)


# ------------------------------------------------------------
# 7. Compare experiments
# ------------------------------------------------------------

experiment_specs = {

    "Baseline (405 features)":
        best_cols,

    "Baseline + 2-hop (525 features)":
        best_cols + two_hop_cols

}

results = []

for name, cols in experiment_specs.items():

    model = make_classifier()

    model.fit(
        combined_train_2hop.loc[
            train_part.index, cols
        ],
        train_part['label'],
        sample_weight=make_sample_weights(
            train_part['label']
        )
    )

    prob = model.predict_proba(
        combined_train_2hop.loc[
            valid_part.index, cols
        ]
    )[:, 1]

    pred = (prob >= 0.5).astype(int)

    results.append({

        "model": name,

        "pr_auc": average_precision_score(
            valid_part['label'],
            prob
        ),

        "roc_auc": roc_auc_score(
            valid_part['label'],
            prob
        ),

        "precision": precision_score(
            valid_part['label'],
            pred,
            zero_division=0
        ),

        "recall": recall_score(
            valid_part['label'],
            pred,
            zero_division=0
        )

    })

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        "roc_auc",
        ascending=False
    )
)

In [ ]:
# ============================================================
# Section 28b — Generate 2-hop Kaggle Submission
# ============================================================

final_cols = best_cols + two_hop_cols

final_model_2hop = make_classifier()

final_model_2hop.fit(
    combined_train_2hop.loc[labelled.index, final_cols],
    labelled["label"],
    sample_weight=make_sample_weights(labelled["label"]),
)

test_prob_2hop = final_model_2hop.predict_proba(
    combined_test_2hop[final_cols]
)[:, 1]

submission_2hop = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

assert len(submission_2hop) == len(test_prob_2hop)

submission_2hop["target"] = test_prob_2hop

output_path = DATA_DIR / "submission_2hop.csv"

submission_2hop.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Prediction mean:", test_prob_2hop.mean())
print("Prediction std:", test_prob_2hop.std())

In [ ]:
# ============================================================
# Section 30 — Validate 2-Hop Ensemble Before Submission
# ============================================================

from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import rankdata, norm
from sklearn.ensemble import HistGradientBoostingClassifier
import lightgbm as lgb
import xgboost as xgb
import numpy as np
import pandas as pd

cols = best_cols + two_hop_cols

X_tr = combined_train_2hop.loc[train_part.index, cols]
y_tr = train_part['label']

X_val = combined_train_2hop.loc[valid_part.index, cols]
y_val = valid_part['label']

scale_pos = (
    (y_tr == 0).sum() /
    max((y_tr == 1).sum(), 1)
)

models = {
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.08,
        num_leaves=31,
        scale_pos_weight=scale_pos,
        random_state=42,
        verbose=-1
    ),

    "XGBoost": xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.08,
        max_depth=6,
        scale_pos_weight=scale_pos,
        eval_metric="aucpr",
        tree_method="hist",
        random_state=42,
        verbosity=0
    ),

    "HistGB": HistGradientBoostingClassifier(
        max_iter=180,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )
}

val_probs = {}

for name, model in models.items():

    print("Training:", name)

    if name == "HistGB":
        model.fit(
            X_tr,
            y_tr,
            sample_weight=make_sample_weights(y_tr)
        )
    else:
        model.fit(X_tr, y_tr)

    val_probs[name] = model.predict_proba(X_val)[:, 1]


def rank_probit(prob):
    ranks = rankdata(prob, method="average") / (len(prob) + 1)
    return norm.ppf(ranks)


z_lgb = rank_probit(val_probs["LightGBM"])
z_xgb = rank_probit(val_probs["XGBoost"])
z_hgb = rank_probit(val_probs["HistGB"])

# Equal-weight ensemble
equal_blend = (z_lgb + z_xgb + z_hgb) / 3

# LightGBM-focused ensemble
weighted_blend = (
    0.5 * z_lgb +
    0.35 * z_xgb +
    0.15 * z_hgb
)

evaluation = {
    **val_probs,
    "Equal Rank Ensemble": equal_blend,
    "Weighted Rank Ensemble": weighted_blend
}

results = []

for name, pred in evaluation.items():

    results.append({
        "model": name,
        "roc_auc": roc_auc_score(y_val, pred),
        "pr_auc": average_precision_score(y_val, pred)
    })

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        "roc_auc",
        ascending=False
    )
)

In [ ]:
import pandas as pd
import numpy as np

datasets = {
    "Train": combined_train_2hop.loc[train_part.index],
    "Validation": combined_train_2hop.loc[valid_part.index],
    "Test": combined_test_2hop
}

rows = []

for name, df in datasets.items():

    row = {
        "dataset": name,
        "n_samples": len(df),
        "time_min": df["time_step"].min(),
        "time_max": df["time_step"].max(),
        "time_mean": df["time_step"].mean(),
    }

    for col in ["in_degree", "out_degree", "total_degree"]:
        if col in df.columns:
            row[f"{col}_mean"] = df[col].mean()
            row[f"{col}_zero_pct"] = (df[col] == 0).mean()

    row["1hop_all_zero_pct"] = (
        df[rich_neighbor_cols].fillna(0).eq(0).all(axis=1).mean()
    )

    row["2hop_all_zero_pct"] = (
        df[two_hop_cols].fillna(0).eq(0).all(axis=1).mean()
    )

    rows.append(row)

diagnostics = pd.DataFrame(rows)
display(diagnostics.T)

In [ ]:
# ============================================================
# Diagnose Train / Validation / Test Distribution
# ============================================================

import pandas as pd
import numpy as np

datasets = {
    "Train": combined_train_2hop.loc[train_part.index],
    "Validation": combined_train_2hop.loc[valid_part.index],
    "Kaggle Test": combined_test_2hop
}

rows = []

for name, df in datasets.items():

    row = {
        "dataset": name,
        "n_samples": len(df),
        "time_min": df["time_step"].min(),
        "time_max": df["time_step"].max(),
        "time_mean": df["time_step"].mean(),
        "time_median": df["time_step"].median(),
    }

    for col in ["in_degree", "out_degree", "total_degree"]:
        if col in df.columns:
            row[f"{col}_mean"] = df[col].mean()
            row[f"{col}_zero_pct"] = (df[col] == 0).mean()

    row["1hop_all_zero_pct"] = (
        df[rich_neighbor_cols]
        .fillna(0)
        .eq(0)
        .all(axis=1)
        .mean()
    )

    row["2hop_all_zero_pct"] = (
        df[two_hop_cols]
        .fillna(0)
        .eq(0)
        .all(axis=1)
        .mean()
    )

    rows.append(row)

diagnostics = pd.DataFrame(rows)

display(diagnostics.round(4))

In [ ]:
train_time = (
    combined_train_2hop.loc[train_part.index, "time_step"]
    .value_counts()
    .sort_index()
)

valid_time = (
    combined_train_2hop.loc[valid_part.index, "time_step"]
    .value_counts()
    .sort_index()
)

test_time = (
    combined_test_2hop["time_step"]
    .value_counts()
    .sort_index()
)

time_distribution = pd.DataFrame({
    "train": train_time,
    "validation": valid_time,
    "test": test_time
}).fillna(0).astype(int)

display(time_distribution)

In [ ]:
# ============================================================
# Section 31 — Diagnose Missing Neighbor Features
# ============================================================

import pandas as pd
import numpy as np

# 1. Identify test nodes with all-zero neighbor features
test_zero_mask = (
    combined_test_2hop[rich_neighbor_cols]
    .fillna(0)
    .eq(0)
    .all(axis=1)
)

zero_nodes = (
    combined_test_2hop.loc[test_zero_mask, 'txId']
    .astype(str)
)

print("Total test nodes:", len(combined_test_2hop))
print("Zero neighbor feature nodes:", len(zero_nodes))
print("Zero percentage:", test_zero_mask.mean())

# 2. Check whether these nodes exist in graph
graph_nodes = set(map(str, G.nodes()))

print(
    "Zero-feature nodes missing from graph:",
    (~zero_nodes.isin(graph_nodes)).sum()
)

# 3. Check whether these nodes exist in aggregation lookup
lookup_nodes = set(node_lookup_k.index.astype(str))

print(
    "Zero-feature nodes missing from lookup:",
    (~zero_nodes.isin(lookup_nodes)).sum()
)

# 4. Check whether their neighbors exist in lookup
edge_sources = edges_str['txId1'].astype(str)
edge_targets = edges_str['txId2'].astype(str)

pred_edges = edges_str[
    edges_str['txId2'].isin(zero_nodes)
]

succ_edges = edges_str[
    edges_str['txId1'].isin(zero_nodes)
]

print("\nEdges involving zero-feature test nodes:")
print("Incoming edges:", len(pred_edges))
print("Outgoing edges:", len(succ_edges))

print(
    "Incoming neighbors in lookup:",
    pred_edges['txId1'].isin(lookup_nodes).sum()
)

print(
    "Outgoing neighbors in lookup:",
    succ_edges['txId2'].isin(lookup_nodes).sum()
)

# 5. Check aggregation feature coverage
print("\nAggregation coverage:")

print(
    "Zero nodes present in rich_neighbor_features:",
    zero_nodes.isin(
        rich_neighbor_features.index.astype(str)
    ).sum()
)

print(
    "Zero nodes present in two_hop_features:",
    zero_nodes.isin(
        two_hop_features.index.astype(str)
    ).sum()
)

# 6. Check prediction alignment
print("\nPrediction alignment:")

assert len(test) == len(combined_test_2hop)

assert np.array_equal(
    test['txId'].astype(str).to_numpy(),
    combined_test_2hop['txId'].astype(str).to_numpy()
)

print("PASSED")

#17 GRAPH STRUCTURAL FEATURES + TIME DECAY

In [ ]:
# ============================================================
# SECTION 46 — GRAPH STRUCTURAL FEATURES + TIME DECAY
# BT4012 Bitcoin Fraud Detection
#
# Current best Public ROC-AUC: 0.95583
#
# Required notebook variables:
#   combined_train_2hop
#   combined_test_2hop
#   train_part
#   valid_part
#   best_cols
#   two_hop_cols
#   edges_str
#
# Models:
#   M1: Baseline 525 + Decay .02
#   M2: Baseline + Basic Graph + Decay .02
#   M3: Baseline + Advanced Graph + Decay .02
#   M4: Baseline + All Graph + Decay .02
#   M5: Baseline / All Graph Blend
#
# Output:
#   submission_s46_basic_graph.csv
#   submission_s46_advanced_graph.csv
#   submission_s46_all_graph.csv
#   submission_s46_graph_blend.csv
#
# ============================================================

import numpy as np
import pandas as pd
import networkx as nx
import warnings
import gc

from pathlib import Path
from scipy.stats import rankdata
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

SEED = 42
DECAY = 0.02

WINDOWS = [
    (14, 15, 21),
    (21, 22, 28),
    (28, 29, 35)
]

OUTPUT_DIR = Path(
    globals().get("DATA_DIR", Path.cwd())
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. BASELINE DATA
# ============================================================

BASE_COLS = list(dict.fromkeys(best_cols))

TWO_HOP_COLS = [
    c for c in dict.fromkeys(two_hop_cols)
    if c not in BASE_COLS
]

FULL_COLS = BASE_COLS + TWO_HOP_COLS

assert len(FULL_COLS) == 525, (
    f"Expected 525 features, got {len(FULL_COLS)}"
)

all_features = pd.concat(
    [combined_train_2hop, combined_test_2hop],
    ignore_index=True
).copy()

all_features["txId"] = all_features["txId"].astype(str)

all_features = (
    all_features
    .drop_duplicates("txId")
    .set_index("txId")
)

labelled = pd.concat(
    [train_part, valid_part],
    ignore_index=True
).copy()

labelled["txId"] = labelled["txId"].astype(str)

assert labelled["txId"].is_unique

X_all = all_features.loc[
    labelled["txId"], FULL_COLS
].reset_index(drop=True)

X_all = X_all.replace(
    [np.inf, -np.inf], np.nan
)

y_all = labelled["label"].astype(int).to_numpy()
t_all = labelled["time_step"].astype(int).to_numpy()
ids_all = labelled["txId"].to_numpy()

test_original = combined_test_2hop.copy()
test_original["txId"] = test_original["txId"].astype(str)

assert test_original["txId"].is_unique

X_test = all_features.loc[
    test_original["txId"], FULL_COLS
].reset_index(drop=True)

X_test = X_test.replace(
    [np.inf, -np.inf], np.nan
)

ids_test = test_original["txId"].to_numpy()

print("=" * 70)
print("SECTION 46 — DATA OVERVIEW")
print("=" * 70)

print(f"Labelled: {len(X_all):,}")
print(f"Test: {len(X_test):,}")
print(f"Baseline features: {len(FULL_COLS)}")

# ============================================================
# 2. PREPARE GRAPH
# ============================================================

print("\nBuilding transaction graph...")

edges = edges_str.copy()

if {"txId1", "txId2"}.issubset(edges.columns):
    SRC_COL, DST_COL = "txId1", "txId2"
elif {"source", "target"}.issubset(edges.columns):
    SRC_COL, DST_COL = "source", "target"
else:
    SRC_COL, DST_COL = edges.columns[:2]

edges = edges[[SRC_COL, DST_COL]].copy()
edges.columns = ["src", "dst"]

edges["src"] = edges["src"].astype(str)
edges["dst"] = edges["dst"].astype(str)

edges = edges.drop_duplicates()

G = nx.from_pandas_edgelist(
    edges,
    source="src",
    target="dst",
    create_using=nx.DiGraph()
)

# Include nodes that have no edges
G.add_nodes_from(ids_all)
G.add_nodes_from(ids_test)

print(f"Graph nodes: {G.number_of_nodes():,}")
print(f"Graph edges: {G.number_of_edges():,}")

# ============================================================
# 3. BASIC GRAPH FEATURES
# ============================================================

print("\nComputing basic graph features...")

nodes = list(G.nodes())

in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

basic = pd.DataFrame(index=nodes)

basic["g_in_degree"] = pd.Series(in_degree)
basic["g_out_degree"] = pd.Series(out_degree)

basic["g_total_degree"] = (
    basic["g_in_degree"] +
    basic["g_out_degree"]
)

basic["g_in_out_ratio"] = (
    basic["g_in_degree"] /
    (basic["g_out_degree"] + 1)
)

basic["g_degree_difference"] = (
    basic["g_in_degree"] -
    basic["g_out_degree"]
)

basic["g_log_total_degree"] = np.log1p(
    basic["g_total_degree"]
)

BASIC_COLS = list(basic.columns)

print(f"Basic graph features: {len(BASIC_COLS)}")

# ============================================================
# 4. ADVANCED GRAPH FEATURES
# ============================================================

print("\nComputing advanced graph features...")

advanced = pd.DataFrame(index=nodes)

# ------------------------------------------------------------
# PageRank
# ------------------------------------------------------------

print("Computing PageRank...")

pagerank = nx.pagerank(
    G,
    alpha=0.85,
    max_iter=100,
    tol=1e-6
)

advanced["g_pagerank"] = pd.Series(pagerank)

# ------------------------------------------------------------
# Weakly Connected Components
# ------------------------------------------------------------

print("Computing connected components...")

component_size = {}

for component in nx.weakly_connected_components(G):

    size = len(component)

    for node in component:
        component_size[node] = size

advanced["g_component_size"] = pd.Series(
    component_size
)

advanced["g_log_component_size"] = np.log1p(
    advanced["g_component_size"]
)

# ------------------------------------------------------------
# Clustering Coefficient
# ------------------------------------------------------------

print("Computing clustering coefficient...")

G_undirected = G.to_undirected()

clustering = nx.clustering(G_undirected)

advanced["g_clustering"] = pd.Series(clustering)

# ------------------------------------------------------------
# Neighbor Degree Statistics
# ------------------------------------------------------------

print("Computing neighbor degree statistics...")

degree = dict(G_undirected.degree())

neighbor_degree_mean = {}
neighbor_degree_max = {}

for node in G_undirected.nodes():

    neighbors = list(G_undirected.neighbors(node))

    if neighbors:

        values = [
            degree[n] for n in neighbors
        ]

        neighbor_degree_mean[node] = np.mean(values)
        neighbor_degree_max[node] = np.max(values)

    else:

        neighbor_degree_mean[node] = 0
        neighbor_degree_max[node] = 0

advanced["g_neighbor_degree_mean"] = pd.Series(
    neighbor_degree_mean
)

advanced["g_neighbor_degree_max"] = pd.Series(
    neighbor_degree_max
)

# ------------------------------------------------------------
# Two-Hop Reachability
# ------------------------------------------------------------

print("Computing two-hop reachability...")

two_hop_count = {}

for node in G_undirected.nodes():

    one_hop = set(
        G_undirected.neighbors(node)
    )

    two_hop = set()

    for neighbor in one_hop:

        two_hop.update(
            G_undirected.neighbors(neighbor)
        )

    two_hop.discard(node)
    two_hop -= one_hop

    two_hop_count[node] = len(two_hop)

advanced["g_two_hop_reachability"] = pd.Series(
    two_hop_count
)

ADVANCED_COLS = list(advanced.columns)

print(f"Advanced graph features: {len(ADVANCED_COLS)}")

# ============================================================
# 5. COMBINE GRAPH FEATURES
# ============================================================

graph_features = basic.join(
    advanced,
    how="outer"
)

graph_features = graph_features.fillna(0)

GRAPH_COLS = BASIC_COLS + ADVANCED_COLS

print("\nGraph feature summary:")

print(
    graph_features[GRAPH_COLS]
    .describe()
    .round(3)
)

# ============================================================
# 6. ALIGN GRAPH FEATURES
# ============================================================

graph_train = (
    graph_features
    .reindex(ids_all)
    .reset_index(drop=True)
    .fillna(0)
)

graph_test = (
    graph_features
    .reindex(ids_test)
    .reset_index(drop=True)
    .fillna(0)
)

X_train_full = pd.concat(
    [X_all, graph_train],
    axis=1
)

X_test_full = pd.concat(
    [X_test, graph_test],
    axis=1
)

MODEL_FEATURES = {
    "M1_Baseline": FULL_COLS,

    "M2_BasicGraph":
        FULL_COLS + BASIC_COLS,

    "M3_AdvancedGraph":
        FULL_COLS + ADVANCED_COLS,

    "M4_AllGraph":
        FULL_COLS + GRAPH_COLS
}

print("\nFeature counts:")

for name, cols in MODEL_FEATURES.items():
    print(f"{name}: {len(cols)}")

# ============================================================
# 7. MODEL FUNCTIONS
# ============================================================

def make_model():

    return LGBMClassifier(
        n_estimators=200,
        learning_rate=0.08,
        num_leaves=31,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    )

def time_weights(times, end_time):

    w = np.exp(
        -DECAY * (end_time - times)
    )

    return w / w.mean()

def evaluate(y, pred):

    return {
        "ROC-AUC": roc_auc_score(y, pred),
        "PR-AUC": average_precision_score(y, pred)
    }

# ============================================================
# 8. ROLLING TEMPORAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("SECTION 46 — ROLLING TEMPORAL VALIDATION")
print("=" * 70)

results = []
rolling_predictions = {}

for train_end, valid_start, valid_end in WINDOWS:

    window_name = f"{valid_start}-{valid_end}"

    print(
        f"\nTrain <= {train_end} | "
        f"Valid {window_name}"
    )

    train_mask = t_all <= train_end

    valid_mask = (
        (t_all >= valid_start) &
        (t_all <= valid_end)
    )

    y_train = y_all[train_mask]
    y_valid = y_all[valid_mask]

    weights = time_weights(
        t_all[train_mask],
        train_end
    )

    rolling_predictions[window_name] = {
        "y_true": y_valid
    }

    for model_name, cols in MODEL_FEATURES.items():

        model = make_model()

        model.fit(
            X_train_full.loc[
                train_mask, cols
            ],
            y_train,
            sample_weight=weights
        )

        pred = model.predict_proba(
            X_train_full.loc[
                valid_mask, cols
            ]
        )[:, 1]

        rolling_predictions[
            window_name
        ][model_name] = pred

        metrics = evaluate(
            y_valid, pred
        )

        results.append({
            "Window": window_name,
            "Model": model_name,
            **metrics
        })

        print(
            f"{model_name:<22} "
            f"AUC={metrics['ROC-AUC']:.6f} "
            f"PR={metrics['PR-AUC']:.6f}"
        )

        del model
        gc.collect()

    # --------------------------------------------------------
    # M5: Conservative probability blend
    # --------------------------------------------------------

    pred_blend = (
        0.75 *
        rolling_predictions[window_name]["M1_Baseline"]
        +
        0.25 *
        rolling_predictions[window_name]["M4_AllGraph"]
    )

    rolling_predictions[
        window_name
    ]["M5_GraphBlend"] = pred_blend

    metrics = evaluate(
        y_valid, pred_blend
    )

    results.append({
        "Window": window_name,
        "Model": "M5_GraphBlend",
        **metrics
    })

    print(
        f"{'M5_GraphBlend':<22} "
        f"AUC={metrics['ROC-AUC']:.6f} "
        f"PR={metrics['PR-AUC']:.6f}"
    )

# ============================================================
# 9. VALIDATION RESULTS
# ============================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("SECTION 46 — ROC-AUC BY WINDOW")
print("=" * 70)

display(
    results_df.pivot(
        index="Window",
        columns="Model",
        values="ROC-AUC"
    ).round(6)
)

summary = (
    results_df
    .groupby("Model")
    .agg(
        Mean_AUC=("ROC-AUC", "mean"),
        Std_AUC=("ROC-AUC", "std"),
        Min_AUC=("ROC-AUC", "min"),
        Mean_PR=("PR-AUC", "mean")
    )
    .sort_values(
        ["Mean_AUC", "Min_AUC"],
        ascending=False
    )
)

print("\n=== ROBUSTNESS SUMMARY ===")

display(summary.round(6))

# ============================================================
# 10. FINAL TRAINING
# ============================================================

print("\n" + "=" * 70)
print("SECTION 46 — FINAL TRAINING")
print("=" * 70)

final_weights = time_weights(
    t_all,
    int(t_all.max())
)

final_predictions = {}

for model_name, cols in MODEL_FEATURES.items():

    model = make_model()

    model.fit(
        X_train_full[cols],
        y_all,
        sample_weight=final_weights
    )

    pred = model.predict_proba(
        X_test_full[cols]
    )[:, 1]

    final_predictions[model_name] = pred

    print(f"{model_name} trained.")

    del model
    gc.collect()

# Blend
final_predictions["M5_GraphBlend"] = (
    0.75 * final_predictions["M1_Baseline"]
    +
    0.25 * final_predictions["M4_AllGraph"]
)

# ============================================================
# 11. SAVE SUBMISSIONS
# ============================================================

print("\n" + "=" * 70)
print("SAVING SECTION 46 SUBMISSIONS")
print("=" * 70)

SUBMISSIONS = {
    "submission_s46_basic_graph.csv":
        "M2_BasicGraph",

    "submission_s46_advanced_graph.csv":
        "M3_AdvancedGraph",

    "submission_s46_all_graph.csv":
        "M4_AllGraph",

    "submission_s46_graph_blend.csv":
        "M5_GraphBlend"
}

for filename, model_name in SUBMISSIONS.items():

    pred = np.asarray(
        final_predictions[model_name]
    )

    assert len(pred) == len(ids_test)
    assert np.isfinite(pred).all()

    submission = pd.DataFrame({
        "txId": ids_test,
        "target": pred
    })

    path = OUTPUT_DIR / filename

    submission.to_csv(
        path,
        index=False
    )

    print(f"Saved: {path}")

# ============================================================
# 12. SAVE NOTEBOOK VARIABLES
# ============================================================

section46_results = results_df
section46_summary = summary
section46_graph_features = graph_features
section46_final_predictions = final_predictions

print("\nSection 46 completed.")
print("Current best Public ROC-AUC: 0.95583")

In [ ]:
import pandas as pd
import numpy as np

# Read the original sample submission to get the correct column names and row ordering
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print("Sample submission columns:", list(sample_sub.columns))
print("Sample submission head:")
display(sample_sub.head())

# Standardize alignment: Map 'test_original' txId predictions to 'index'
# Let's verify that test_original indices align with sample_sub
assert len(sample_sub) == len(test_original), f"Length mismatch: {len(sample_sub)} vs {len(test_original)}"

# Ensure the output maps exactly to the ID expected by the competition platform
id_column = sample_sub.columns[0]
target_column = sample_sub.columns[1]
print(f"Correct ID column: '{id_column}', Correct target column: '{target_column}'")

# Regenerate Section 46 submissions with the correct index column alignment
SUBMISSIONS_FIXED = {
    "submission_s46_basic_graph.csv": "M2_BasicGraph",
    "submission_s46_advanced_graph.csv": "M3_AdvancedGraph",
    "submission_s46_all_graph.csv": "M4_AllGraph",
    "submission_s46_graph_blend.csv": "M5_GraphBlend"
}

for filename, model_name in SUBMISSIONS_FIXED.items():
    pred = np.asarray(section46_final_predictions[model_name])

    # Build standard submission DataFrame aligning directly with sample_submission index
    submission_df = pd.DataFrame({
        id_column: sample_sub[id_column].values,
        target_column: pred
    })

    path = OUTPUT_DIR / filename
    submission_df.to_csv(path, index=False)
    print(f"Successfully saved correctly formatted submission to: {path}")
    print(f"Preview:")
    display(submission_df.head())


#18 — GRAPH FEATURE SELECTION + LGBM OPTIMIZATION

In [ ]:
# ============================================================
# SECTION 47 — GRAPH FEATURE SELECTION + LGBM OPTIMIZATION
# BT4012 Bitcoin Fraud Detection
#
# Current best Public ROC-AUC: 0.95613
# Best model: 538 Features + LGBM + Decay 0.02
#
# Required existing variables:
#   X_train_full        (31235 x 538)
#   X_test_full         (15329 x 538)
#   y_all
#   t_all
#   ids_test
#   FULL_COLS           (525 columns)
#   GRAPH_COLS          (13 columns)
#   OUTPUT_DIR
#
# Outputs:
#   submission_s47_selected_graph.csv
#   submission_s47_regularized.csv
#   submission_s47_conservative.csv
#   submission_s47_feature_subsample.csv
#   submission_s47_more_trees.csv
#   submission_s47_best_blend.csv
#
# ============================================================

import numpy as np
import pandas as pd
import gc
import warnings

from pathlib import Path

from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

SEED = 42
DECAY = 0.02

WINDOWS = [
    (14, 15, 21),
    (21, 22, 28),
    (28, 29, 35)
]

OUTPUT_DIR = Path(
    globals().get("OUTPUT_DIR", Path.cwd())
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. DATA CHECK
# ============================================================

assert len(FULL_COLS) == 525
assert len(GRAPH_COLS) == 13
assert len(X_train_full) == len(y_all)
assert len(X_test_full) == len(ids_test)

ALL_COLS = FULL_COLS + GRAPH_COLS

assert len(ALL_COLS) == 538
assert len(set(ALL_COLS)) == 538

X_train_full = X_train_full[ALL_COLS].replace(
    [np.inf, -np.inf], np.nan
)

X_test_full = X_test_full[ALL_COLS].replace(
    [np.inf, -np.inf], np.nan
)

y_all = np.asarray(y_all)
t_all = np.asarray(t_all)

print("=" * 70)
print("SECTION 47 — DATA OVERVIEW")
print("=" * 70)

print("Train:", X_train_full.shape)
print("Test:", X_test_full.shape)
print("Graph features:", GRAPH_COLS)

# ============================================================
# 2. MODEL CONFIGURATIONS
# ============================================================

BASE_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.08,
    "num_leaves": 31,
    "random_state": SEED,
    "n_jobs": -1,
    "verbose": -1
}

MODEL_CONFIGS = {

    "M1_Baseline538": {},

    "M3_Regularized": {
        "min_child_samples": 40,
        "reg_lambda": 1.0
    },

    "M4_Conservative": {
        "num_leaves": 15,
        "min_child_samples": 40
    },

    "M5_FeatureSubsample": {
        "colsample_bytree": 0.85,
        "reg_lambda": 1.0
    },

    "M6_MoreTrees": {
        "n_estimators": 300,
        "learning_rate": 0.05
    }
}

def make_model(overrides=None):

    params = BASE_PARAMS.copy()

    if overrides:
        params.update(overrides)

    return LGBMClassifier(**params)


def time_weights(times, end_time):

    w = np.exp(
        -DECAY * (end_time - times)
    )

    return w / w.mean()


def evaluate(y, pred):

    return {
        "ROC-AUC": roc_auc_score(y, pred),
        "PR-AUC": average_precision_score(y, pred)
    }

# ============================================================
# 3. GRAPH FEATURE SELECTION
# ============================================================

def select_graph_features(
    X_train,
    y_train,
    weights,
    top_k=7
):

    selector = make_model()

    selector.fit(
        X_train[ALL_COLS],
        y_train,
        sample_weight=weights
    )

    importance = pd.Series(
        selector.feature_importances_,
        index=ALL_COLS
    )

    graph_importance = (
        importance[GRAPH_COLS]
        .sort_values(ascending=False)
    )

    selected = (
        graph_importance
        .head(top_k)
        .index
        .tolist()
    )

    return selected, graph_importance

# ============================================================
# 4. ROLLING VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("SECTION 47 — ROLLING VALIDATION")
print("=" * 70)

results = []
rolling_predictions = {}
selected_features_by_window = {}

for train_end, valid_start, valid_end in WINDOWS:

    window_name = f"{valid_start}-{valid_end}"

    print(
        f"\nTrain <= {train_end} | "
        f"Valid {window_name}"
    )

    train_mask = t_all <= train_end

    valid_mask = (
        (t_all >= valid_start) &
        (t_all <= valid_end)
    )

    X_tr = X_train_full.loc[
        train_mask
    ].reset_index(drop=True)

    X_va = X_train_full.loc[
        valid_mask
    ].reset_index(drop=True)

    y_tr = y_all[train_mask]
    y_va = y_all[valid_mask]

    weights = time_weights(
        t_all[train_mask],
        train_end
    )

    rolling_predictions[window_name] = {
        "y_true": y_va
    }

    # --------------------------------------------------------
    # M2: Select top graph features using training only
    # --------------------------------------------------------

    selected, graph_importance = (
        select_graph_features(
            X_tr,
            y_tr,
            weights,
            top_k=7
        )
    )

    selected_features_by_window[
        window_name
    ] = selected

    print("Selected graph features:", selected)

    selected_cols = FULL_COLS + selected

    model_selected = make_model()

    model_selected.fit(
        X_tr[selected_cols],
        y_tr,
        sample_weight=weights
    )

    pred_selected = model_selected.predict_proba(
        X_va[selected_cols]
    )[:, 1]

    rolling_predictions[
        window_name
    ]["M2_SelectedGraph"] = pred_selected

    metrics = evaluate(y_va, pred_selected)

    results.append({
        "Window": window_name,
        "Model": "M2_SelectedGraph",
        **metrics
    })

    print(
        f"M2_SelectedGraph       "
        f"AUC={metrics['ROC-AUC']:.6f}"
    )

    del model_selected

    # --------------------------------------------------------
    # Other models
    # --------------------------------------------------------

    for name, overrides in MODEL_CONFIGS.items():

        model = make_model(overrides)

        model.fit(
            X_tr[ALL_COLS],
            y_tr,
            sample_weight=weights
        )

        pred = model.predict_proba(
            X_va[ALL_COLS]
        )[:, 1]

        rolling_predictions[
            window_name
        ][name] = pred

        metrics = evaluate(y_va, pred)

        results.append({
            "Window": window_name,
            "Model": name,
            **metrics
        })

        print(
            f"{name:<24} "
            f"AUC={metrics['ROC-AUC']:.6f}"
        )

        del model
        gc.collect()

# ============================================================
# 5. VALIDATION SUMMARY
# ============================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("SECTION 47 — ROC-AUC BY WINDOW")
print("=" * 70)

display(
    results_df.pivot(
        index="Window",
        columns="Model",
        values="ROC-AUC"
    ).round(6)
)

summary = (
    results_df
    .groupby("Model")
    .agg(
        Mean_AUC=("ROC-AUC", "mean"),
        Min_AUC=("ROC-AUC", "min"),
        Std_AUC=("ROC-AUC", "std"),
        Mean_PR=("PR-AUC", "mean")
    )
    .sort_values(
        ["Mean_AUC", "Min_AUC"],
        ascending=False
    )
)

print("\n=== ROBUSTNESS SUMMARY ===")

display(summary.round(6))

# ============================================================
# 6. SELECT BEST NON-BASELINE MODEL
# ============================================================

candidate_summary = summary.drop(
    index="M1_Baseline538"
)

best_candidate = (
    candidate_summary.index[0]
)

print("\nBest candidate:", best_candidate)

# ============================================================
# 7. FINAL FEATURE SELECTION
# ============================================================

final_weights = time_weights(
    t_all,
    int(t_all.max())
)

final_selected, final_graph_importance = (
    select_graph_features(
        X_train_full,
        y_all,
        final_weights,
        top_k=7
    )
)

FINAL_SELECTED_COLS = (
    FULL_COLS + final_selected
)

print("\nFinal selected graph features:")

print(final_selected)

print("\nGraph feature importance:")

display(
    final_graph_importance.to_frame(
        "Importance"
    )
)

# ============================================================
# 8. FINAL TRAINING
# ============================================================

print("\n" + "=" * 70)
print("SECTION 47 — FINAL TRAINING")
print("=" * 70)

final_predictions = {}

# ------------------------------------------------------------
# M2 selected features
# ------------------------------------------------------------

model = make_model()

model.fit(
    X_train_full[FINAL_SELECTED_COLS],
    y_all,
    sample_weight=final_weights
)

final_predictions["M2_SelectedGraph"] = (
    model.predict_proba(
        X_test_full[FINAL_SELECTED_COLS]
    )[:, 1]
)

del model
gc.collect()

# ------------------------------------------------------------
# M1, M3, M4, M5, M6
# ------------------------------------------------------------

for name, overrides in MODEL_CONFIGS.items():

    model = make_model(overrides)

    model.fit(
        X_train_full[ALL_COLS],
        y_all,
        sample_weight=final_weights
    )

    final_predictions[name] = (
        model.predict_proba(
            X_test_full[ALL_COLS]
        )[:, 1]
    )

    print(f"{name} trained.")

    del model
    gc.collect()

# ============================================================
# 9. BEST CANDIDATE BLEND
# ============================================================

baseline_pred = (
    final_predictions["M1_Baseline538"]
)

candidate_pred = (
    final_predictions[best_candidate]
)

final_predictions["M7_BestBlend"] = (
    0.75 * baseline_pred
    +
    0.25 * candidate_pred
)

# ============================================================
# 10. SAVE SUBMISSIONS
# ============================================================

SUBMISSIONS = {

    "submission_s47_selected_graph.csv":
        "M2_SelectedGraph",

    "submission_s47_regularized.csv":
        "M3_Regularized",

    "submission_s47_conservative.csv":
        "M4_Conservative",

    "submission_s47_feature_subsample.csv":
        "M5_FeatureSubsample",

    "submission_s47_more_trees.csv":
        "M6_MoreTrees",

    "submission_s47_best_blend.csv":
        "M7_BestBlend"
}

print("\n" + "=" * 70)
print("SAVING SECTION 47 SUBMISSIONS")
print("=" * 70)

for filename, name in SUBMISSIONS.items():

    pred = np.asarray(
        final_predictions[name]
    )

    assert len(pred) == len(ids_test)
    assert np.isfinite(pred).all()

    submission = pd.DataFrame({
        "txId": ids_test,
        "target": pred
    })

    path = OUTPUT_DIR / filename

    submission.to_csv(
        path,
        index=False
    )

    print(f"Saved: {path}")

# ============================================================
# 11. SAVE NOTEBOOK VARIABLES
# ============================================================

section47_results = results_df
section47_summary = summary
section47_predictions = rolling_predictions
section47_final_predictions = final_predictions
section47_selected_features = final_selected
section47_best_candidate = best_candidate

print("\nSection 47 completed.")
print("Current best Public ROC-AUC: 0.95613")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load sample_submission to get the correct column structure
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print("Sample submission columns:", list(sample_sub.columns))

id_column = sample_sub.columns[0]     # 'index'
target_column = sample_sub.columns[1] # 'target'

# Quick sanity check on lengths
assert len(sample_sub) == len(ids_test), f"Length mismatch: {len(sample_sub)} vs {len(ids_test)}"

# 2. Re-save Section 47 Submissions using the correct 'index' column structure
SUBMISSIONS_FIXED_S47 = {
    "submission_s47_selected_graph.csv": "M2_SelectedGraph",
    "submission_s47_regularized.csv": "M3_Regularized",
    "submission_s47_conservative.csv": "M4_Conservative",
    "submission_s47_feature_subsample.csv": "M5_FeatureSubsample",
    "submission_s47_more_trees.csv": "M6_MoreTrees",
    "submission_s47_best_blend.csv": "M7_BestBlend"
}

print("\nRegenerating Section 47 submissions with the correct format...")
for filename, model_name in SUBMISSIONS_FIXED_S47.items():
    pred = np.asarray(section47_final_predictions[model_name])

    # Align predictions directly with the exact sequence expected by sample_submission
    submission_df = pd.DataFrame({
        id_column: sample_sub[id_column].values,
        target_column: pred
    })

    path = OUTPUT_DIR / filename
    submission_df.to_csv(path, index=False)
    print(f"Correctly formatted submission saved to: {path}")
    display(submission_df.head(2))


#19 DIRECTED GRAPH FEATURES + VALID SUBMISSIONS

In [ ]:
# ============================================================
# SECTION 48 — DIRECTED GRAPH FEATURES + VALID SUBMISSIONS
# BT4012 Bitcoin Fraud Detection
#
# Current best Public ROC-AUC: 0.95613
#
# IMPORTANT:
# Submission ID column is read from sample_submission.csv.
# Predictions are aligned to the original test.csv row order.
#
# Required variables from Sections 46/47:
#   X_train_full
#   X_test_full
#   y_all
#   t_all
#   ids_all
#   ids_test
#   FULL_COLS
#   GRAPH_COLS
#   edges_str
#   DATA_DIR
#
# Outputs:
#   submission_s48_neighbor.csv
#   submission_s48_multihop.csv
#   submission_s48_all_directed.csv
#   submission_s48_blend.csv
#
# Also repairs Section 46/47 submissions if their
# in-memory prediction arrays are available.
# ============================================================

import numpy as np
import pandas as pd
import networkx as nx
import gc
import warnings

from pathlib import Path
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

SEED = 42
DECAY = 0.02

WINDOWS = [
    (14, 15, 21),
    (21, 22, 28),
    (28, 29, 35)
]

DATA_DIR = Path(DATA_DIR)
OUTPUT_DIR = DATA_DIR

# ============================================================
# 1. READ ORIGINAL TEST + SAMPLE SUBMISSION
# ============================================================

print("=" * 75)
print("SECTION 48 — SUBMISSION SCHEMA VALIDATION")
print("=" * 75)

test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

assert test_path.exists(), f"Missing: {test_path}"
assert sample_path.exists(), f"Missing: {sample_path}"

test_raw = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)

print("\nOriginal test columns:")
print(test_raw.columns.tolist())

print("\nSample submission columns:")
print(sample.columns.tolist())

print("\nSample submission preview:")
display(sample.head())

assert len(test_raw) == len(sample), (
    "test.csv and sample_submission.csv have different row counts. "
    "Cannot assume positional alignment."
)

assert "txId" in test_raw.columns, (
    "test.csv must contain txId for prediction alignment."
)

assert len(sample.columns) == 2, (
    "Expected a two-column sample submission. "
    "Inspect the file before proceeding."
)

SUBMISSION_ID_COL = sample.columns[0]
SUBMISSION_TARGET_COL = sample.columns[1]

print("\nDetected submission schema:")
print("ID column:", SUBMISSION_ID_COL)
print("Target column:", SUBMISSION_TARGET_COL)

# ============================================================
# 2. VERIFY PREDICTION ALIGNMENT
# ============================================================

# IDs used by Sections 46/47 to produce predictions
prediction_ids = pd.Index(
    pd.Series(ids_test).astype(str),
    name="txId"
)

original_test_ids = pd.Index(
    test_raw["txId"].astype(str),
    name="txId"
)

assert prediction_ids.is_unique, (
    "Prediction txId values are not unique."
)

assert original_test_ids.is_unique, (
    "Original test txId values are not unique."
)

assert len(prediction_ids) == len(original_test_ids)

missing_ids = original_test_ids.difference(prediction_ids)
extra_ids = prediction_ids.difference(original_test_ids)

assert len(missing_ids) == 0 and len(extra_ids) == 0, (
    "Prediction IDs do not match original test IDs."
)

# Position of each original test row in the prediction arrays
prediction_positions = prediction_ids.get_indexer(
    original_test_ids
)

assert (prediction_positions >= 0).all()

print("\nPrediction alignment verified.")
print(
    "Prediction order already matches test.csv:",
    prediction_ids.equals(original_test_ids)
)

def save_valid_submission(predictions, filename):
    """
    Align predictions by txId to original test.csv order,
    then use the exact sample_submission.csv ID values.
    """

    predictions = np.asarray(predictions).reshape(-1)

    assert len(predictions) == len(prediction_ids)
    assert np.isfinite(predictions).all()

    aligned_predictions = predictions[
        prediction_positions
    ]

    submission = sample.copy()

    submission[SUBMISSION_TARGET_COL] = aligned_predictions

    # Preserve exact column order and original ID values
    submission = submission[
        sample.columns.tolist()
    ]

    assert len(submission) == len(sample)

    assert submission[
        SUBMISSION_ID_COL
    ].equals(sample[SUBMISSION_ID_COL])

    assert submission.columns.tolist() == sample.columns.tolist()

    output_path = OUTPUT_DIR / filename

    submission.to_csv(
        output_path,
        index=False
    )

    print(f"Saved: {output_path}")

    return submission

# ============================================================
# 3. PREPARE DIRECTED GRAPH
# ============================================================

print("\n" + "=" * 75)
print("BUILDING DIRECTED GRAPH")
print("=" * 75)

edges = edges_str.copy()

if {"txId1", "txId2"}.issubset(edges.columns):
    src_col, dst_col = "txId1", "txId2"
elif {"source", "target"}.issubset(edges.columns):
    src_col, dst_col = "source", "target"
else:
    src_col, dst_col = edges.columns[:2]

edges = edges[[src_col, dst_col]].copy()
edges.columns = ["src", "dst"]

edges["src"] = edges["src"].astype(str)
edges["dst"] = edges["dst"].astype(str)

edges = edges.drop_duplicates()

G = nx.from_pandas_edgelist(
    edges,
    source="src",
    target="dst",
    create_using=nx.DiGraph()
)

G.add_nodes_from(
    pd.Series(ids_all).astype(str)
)

G.add_nodes_from(
    pd.Series(ids_test).astype(str)
)

print("Nodes:", f"{G.number_of_nodes():,}")
print("Edges:", f"{G.number_of_edges():,}")

# ============================================================
# 4. DIRECTED NEIGHBOR FEATURES
# ============================================================

print("\nComputing directed neighbor features...")

nodes = list(G.nodes())

in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

total_degree = {
    node: in_degree[node] + out_degree[node]
    for node in nodes
}

neighbor_features = {}

for i, node in enumerate(nodes):

    predecessors = list(G.predecessors(node))
    successors = list(G.successors(node))

    pred_total = [
        total_degree[n] for n in predecessors
    ]

    succ_total = [
        total_degree[n] for n in successors
    ]

    pred_in = [
        in_degree[n] for n in predecessors
    ]

    pred_out = [
        out_degree[n] for n in predecessors
    ]

    succ_in = [
        in_degree[n] for n in successors
    ]

    succ_out = [
        out_degree[n] for n in successors
    ]

    def safe_mean(values):
        return float(np.mean(values)) if values else 0.0

    def safe_max(values):
        return float(np.max(values)) if values else 0.0

    def safe_std(values):
        return float(np.std(values)) if values else 0.0

    pred_sum = float(sum(pred_total))
    succ_sum = float(sum(succ_total))

    neighbor_features[node] = {

        "dg_pred_degree_mean": safe_mean(pred_total),
        "dg_pred_degree_max": safe_max(pred_total),
        "dg_pred_degree_std": safe_std(pred_total),

        "dg_succ_degree_mean": safe_mean(succ_total),
        "dg_succ_degree_max": safe_max(succ_total),
        "dg_succ_degree_std": safe_std(succ_total),

        "dg_pred_in_degree_mean": safe_mean(pred_in),
        "dg_pred_out_degree_mean": safe_mean(pred_out),

        "dg_succ_in_degree_mean": safe_mean(succ_in),
        "dg_succ_out_degree_mean": safe_mean(succ_out),

        "dg_pred_degree_sum": pred_sum,
        "dg_succ_degree_sum": succ_sum,

        "dg_neighbor_degree_imbalance":
            pred_sum - succ_sum,

        "dg_neighbor_degree_ratio":
            (pred_sum + 1.0) / (succ_sum + 1.0),

        "dg_reciprocal_neighbors":
            len(set(predecessors).intersection(successors))
    }

    if (i + 1) % 50000 == 0:
        print(f"Processed {i + 1:,} nodes")

neighbor_df = pd.DataFrame.from_dict(
    neighbor_features,
    orient="index"
)

NEIGHBOR_COLS = list(neighbor_df.columns)

print("Directed neighbor features:", len(NEIGHBOR_COLS))

# ============================================================
# 5. DIRECTED MULTI-HOP FEATURES
# ============================================================

print("\nComputing directed multi-hop features...")

multihop_features = {}

for i, node in enumerate(nodes):

    pred_1 = set(G.predecessors(node))
    succ_1 = set(G.successors(node))

    # Exact directed 2-hop sets
    pred_2 = set()
    succ_2 = set()

    for n in pred_1:
        pred_2.update(G.predecessors(n))

    for n in succ_1:
        succ_2.update(G.successors(n))

    pred_2.discard(node)
    succ_2.discard(node)

    pred_2 -= pred_1
    succ_2 -= succ_1

    # Directed 3-hop sets
    pred_3 = set()
    succ_3 = set()

    for n in pred_2:
        pred_3.update(G.predecessors(n))

    for n in succ_2:
        succ_3.update(G.successors(n))

    pred_3.discard(node)
    succ_3.discard(node)

    pred_3 -= pred_1
    pred_3 -= pred_2

    succ_3 -= succ_1
    succ_3 -= succ_2

    multihop_features[node] = {

        "dg_pred_2hop_count": len(pred_2),
        "dg_succ_2hop_count": len(succ_2),

        "dg_pred_3hop_count": len(pred_3),
        "dg_succ_3hop_count": len(succ_3),

        "dg_2hop_imbalance":
            len(pred_2) - len(succ_2),

        "dg_3hop_imbalance":
            len(pred_3) - len(succ_3),

        "dg_2hop_ratio":
            (len(pred_2) + 1) /
            (len(succ_2) + 1),

        "dg_3hop_ratio":
            (len(pred_3) + 1) /
            (len(succ_3) + 1),

        "dg_upstream_expansion":
            len(pred_2) / (len(pred_1) + 1),

        "dg_downstream_expansion":
            len(succ_2) / (len(succ_1) + 1)
    }

    if (i + 1) % 50000 == 0:
        print(f"Processed {i + 1:,} nodes")

multihop_df = pd.DataFrame.from_dict(
    multihop_features,
    orient="index"
)

MULTIHOP_COLS = list(multihop_df.columns)

print("Directed multi-hop features:", len(MULTIHOP_COLS))

# ============================================================
# 6. COMBINE AND ALIGN FEATURES
# ============================================================

directed_graph_features = neighbor_df.join(
    multihop_df,
    how="outer"
).fillna(0)

DIRECTED_COLS = NEIGHBOR_COLS + MULTIHOP_COLS

train_ids_str = pd.Series(ids_all).astype(str).to_numpy()
test_ids_str = pd.Series(ids_test).astype(str).to_numpy()

directed_train = (
    directed_graph_features
    .reindex(train_ids_str)
    .reset_index(drop=True)
    .fillna(0)
)

directed_test = (
    directed_graph_features
    .reindex(test_ids_str)
    .reset_index(drop=True)
    .fillna(0)
)

BASE_538_COLS = list(dict.fromkeys(
    list(FULL_COLS) + list(GRAPH_COLS)
))

X_s48_train = pd.concat(
    [
        X_train_full[BASE_538_COLS].reset_index(drop=True),
        directed_train
    ],
    axis=1
)

X_s48_test = pd.concat(
    [
        X_test_full[BASE_538_COLS].reset_index(drop=True),
        directed_test
    ],
    axis=1
)

MODEL_COLS = {

    "M1_Baseline538":
        BASE_538_COLS,

    "M2_DirectedNeighbor":
        BASE_538_COLS + NEIGHBOR_COLS,

    "M3_DirectedMultiHop":
        BASE_538_COLS + MULTIHOP_COLS,

    "M4_AllDirected":
        BASE_538_COLS + DIRECTED_COLS
}

print("\nFeature counts:")

for name, cols in MODEL_COLS.items():
    print(f"{name}: {len(cols)}")

# ============================================================
# 7. MODEL FUNCTIONS
# ============================================================

def make_model():
    return LGBMClassifier(
        n_estimators=200,
        learning_rate=0.08,
        num_leaves=31,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    )

def time_weights(times, end_time):

    weights = np.exp(
        -DECAY * (end_time - times)
    )

    return weights / weights.mean()

# ============================================================
# 8. ROLLING VALIDATION
# ============================================================

print("\n" + "=" * 75)
print("SECTION 48 — ROLLING VALIDATION")
print("=" * 75)

results = []
rolling_predictions = {}

for train_end, valid_start, valid_end in WINDOWS:

    window_name = f"{valid_start}-{valid_end}"

    train_mask = t_all <= train_end

    valid_mask = (
        (t_all >= valid_start) &
        (t_all <= valid_end)
    )

    weights = time_weights(
        t_all[train_mask],
        train_end
    )

    y_tr = y_all[train_mask]
    y_va = y_all[valid_mask]

    rolling_predictions[window_name] = {
        "y_true": y_va
    }

    print(
        f"\nTrain <= {train_end} | "
        f"Valid {window_name}"
    )

    for name, cols in MODEL_COLS.items():

        model = make_model()

        model.fit(
            X_s48_train.loc[train_mask, cols],
            y_tr,
            sample_weight=weights
        )

        pred = model.predict_proba(
            X_s48_train.loc[valid_mask, cols]
        )[:, 1]

        rolling_predictions[
            window_name
        ][name] = pred

        results.append({
            "Window": window_name,
            "Model": name,
            "ROC-AUC": roc_auc_score(y_va, pred),
            "PR-AUC": average_precision_score(y_va, pred)
        })

        print(
            f"{name:<25} "
            f"AUC={roc_auc_score(y_va, pred):.6f}"
        )

        del model
        gc.collect()

    blend = (
        0.75 *
        rolling_predictions[window_name]["M1_Baseline538"]
        +
        0.25 *
        rolling_predictions[window_name]["M4_AllDirected"]
    )

    rolling_predictions[
        window_name
    ]["M5_DirectedBlend"] = blend

    results.append({
        "Window": window_name,
        "Model": "M5_DirectedBlend",
        "ROC-AUC": roc_auc_score(y_va, blend),
        "PR-AUC": average_precision_score(y_va, blend)
    })

    print(
        f"{'M5_DirectedBlend':<25} "
        f"AUC={roc_auc_score(y_va, blend):.6f}"
    )

# ============================================================
# 9. VALIDATION SUMMARY
# ============================================================

section48_results = pd.DataFrame(results)

print("\n" + "=" * 75)
print("SECTION 48 — VALIDATION SUMMARY")
print("=" * 75)

display(
    section48_results.pivot(
        index="Window",
        columns="Model",
        values="ROC-AUC"
    ).round(6)
)

section48_summary = (
    section48_results
    .groupby("Model")
    .agg(
        Mean_AUC=("ROC-AUC", "mean"),
        Min_AUC=("ROC-AUC", "min"),
        Mean_PR=("PR-AUC", "mean")
    )
    .sort_values("Mean_AUC", ascending=False)
)

display(section48_summary.round(6))

# ============================================================
# 10. FINAL TRAINING
# ============================================================

print("\n" + "=" * 75)
print("SECTION 48 — FINAL TRAINING")
print("=" * 75)

final_weights = time_weights(
    t_all,
    int(np.max(t_all))
)

section48_final_predictions = {}

for name, cols in MODEL_COLS.items():

    model = make_model()

    model.fit(
        X_s48_train[cols],
        y_all,
        sample_weight=final_weights
    )

    pred = model.predict_proba(
        X_s48_test[cols]
    )[:, 1]

    section48_final_predictions[name] = pred

    print(f"{name} trained.")

    del model
    gc.collect()

section48_final_predictions["M5_DirectedBlend"] = (
    0.75 *
    section48_final_predictions["M1_Baseline538"]
    +
    0.25 *
    section48_final_predictions["M4_AllDirected"]
)

# ============================================================
# 11. SAVE VALID SECTION 48 SUBMISSIONS
# ============================================================

print("\n" + "=" * 75)
print("SAVING VALID SECTION 48 SUBMISSIONS")
print("=" * 75)

submission_map = {
    "submission_s48_neighbor.csv":
        "M2_DirectedNeighbor",

    "submission_s48_multihop.csv":
        "M3_DirectedMultiHop",

    "submission_s48_all_directed.csv":
        "M4_AllDirected",

    "submission_s48_blend.csv":
        "M5_DirectedBlend"
}

for filename, name in submission_map.items():

    save_valid_submission(
        section48_final_predictions[name],
        filename
    )

# ============================================================
# 12. REPAIR SECTION 46 / 47 SUBMISSIONS
# ============================================================

print("\n" + "=" * 75)
print("REPAIRING PREVIOUS SUBMISSIONS")
print("=" * 75)

# Section 46
if "section46_final_predictions" in globals():

    s46_mapping = {
        "submission_s46_basic_graph_FIXED.csv":
            "M2_BasicGraph",

        "submission_s46_advanced_graph_FIXED.csv":
            "M3_AdvancedGraph",

        "submission_s46_all_graph_FIXED.csv":
            "M4_AllGraph",

        "submission_s46_graph_blend_FIXED.csv":
            "M5_GraphBlend"
    }

    for filename, key in s46_mapping.items():

        if key in section46_final_predictions:

            save_valid_submission(
                section46_final_predictions[key],
                filename
            )

# Section 47
if "section47_final_predictions" in globals():

    s47_mapping = {
        "submission_s47_selected_graph_FIXED.csv":
            "M2_SelectedGraph",

        "submission_s47_best_blend_FIXED.csv":
            "M7_BestBlend"
    }

    for filename, key in s47_mapping.items():

        if key in section47_final_predictions:

            save_valid_submission(
                section47_final_predictions[key],
                filename
            )

# ============================================================
# 13. FINAL CHECK
# ============================================================

print("\n" + "=" * 75)
print("FINAL SUBMISSION CHECK")
print("=" * 75)

check_file = OUTPUT_DIR / "submission_s48_all_directed.csv"

check_df = pd.read_csv(check_file)

assert check_df.columns.tolist() == sample.columns.tolist()
assert len(check_df) == len(sample)

print("Expected columns:", sample.columns.tolist())
print("Generated columns:", check_df.columns.tolist())

print("\nSubmission preview:")
display(check_df.head())

print("\nSection 48 completed successfully.")
print("Current best Public ROC-AUC: 0.95613")

#20

In [ ]:
# ===============================================================
# SECTION 52B — DIRECTED GRAPHSAGE
# Proper Temporal Validation + LightGBM Rank Ensemble
#
# Requirements from previous sections:
#   DATA_DIR
#   X_s48_train, X_s48_test
#   FULL_COLS, GRAPH_COLS
#   y_all, t_all, ids_all, ids_test
#
# Outputs:
#   section52b_validation.csv
#   section52b_summary.csv
#   section52b_epoch_selection.csv
#   section52b_correlations.csv
#
#   submission_s52b_directed.csv
#   submission_s52b_rank_90_10.csv
#   submission_s52b_rank_80_20.csv
#   submission_s52b_rank_70_30.csv
#
# NOTE:
#   Start with N_SEEDS=1 to verify the pipeline.
#   Increase to 3 for robustness after the first run.
# ===============================================================

import sys
import subprocess
import importlib.util

for package in ["torch_geometric", "lightgbm"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "-q",
            "install",
            package
        ])

import gc
import copy
import random
import warnings

from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import SAGEConv

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

from scipy.stats import rankdata, spearmanr
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")


# ================================================================
# 1. CONFIGURATION
# ================================================================

DATA_DIR = Path(DATA_DIR)
OUTPUT_DIR = DATA_DIR

SEED = 42

# First run: 1
# Robustness experiment: 3
N_SEEDS = 1

HIDDEN_DIM = 128
EMBED_DIM = 64

DROPOUT = 0.30

LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 140
PATIENCE = 20
MIN_EPOCHS = 20

DECAY = 0.02

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 75)
print("SECTION 52B — DIRECTED GRAPHSAGE")
print("=" * 75)

print("Device:", DEVICE)
print("Seeds:", N_SEEDS)


def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ================================================================
# 2. LOAD DATA
# ================================================================

train_raw = pd.read_csv(
    DATA_DIR / "train.csv"
)

test_raw = pd.read_csv(
    DATA_DIR / "test.csv"
)

edges_raw = pd.read_csv(
    DATA_DIR / "txs_edgelist.csv"
)

sample = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

assert {"txId", "time_step"}.issubset(train_raw.columns)
assert {"txId", "time_step"}.issubset(test_raw.columns)

assert {"txId1", "txId2"}.issubset(edges_raw.columns)

RAW_FEATURES = [
    f"feat_{i}"
    for i in range(1, 166)
]

FEATURE_COLS = [
    c for c in RAW_FEATURES
    if c in train_raw.columns
    and c in test_raw.columns
]

assert len(FEATURE_COLS) > 0

print("Raw features:", len(FEATURE_COLS))


# ================================================================
# 3. BUILD NODE TABLE
# ================================================================

# IMPORTANT:
# We preserve all graph node IDs, but nodes without
# observable features/time are excluded from message passing.
#
# If a complete unlabeled-node feature table becomes available,
# it should be merged here before graph construction.

all_ids = pd.Index(
    pd.concat(
        [
            train_raw["txId"].astype(str),
            test_raw["txId"].astype(str),
            edges_raw["txId1"].astype(str),
            edges_raw["txId2"].astype(str)
        ],
        ignore_index=True
    ).unique()
)

node_df = pd.DataFrame({
    "txId": all_ids
})

feature_source = pd.concat(
    [
        train_raw[
            ["txId", "time_step"] + FEATURE_COLS
        ],
        test_raw[
            ["txId", "time_step"] + FEATURE_COLS
        ]
    ],
    ignore_index=True
)

feature_source["txId"] = (
    feature_source["txId"].astype(str)
)

feature_source = feature_source.drop_duplicates(
    subset="txId",
    keep="first"
)

node_df = node_df.merge(
    feature_source,
    on="txId",
    how="left"
)

node_times = pd.to_numeric(
    node_df["time_step"],
    errors="coerce"
).to_numpy(dtype=float)

has_observed_time = np.isfinite(node_times)

print("\nGraph node coverage")
print("Total graph nodes:", len(node_df))
print("Nodes with known time/features:",
      has_observed_time.sum())
print("Nodes missing time/features:",
      (~has_observed_time).sum())

if (~has_observed_time).sum() > 0:
    print(
        "\nWARNING: Some unlabeled graph nodes lack "
        "feature rows. They will be excluded from "
        "message passing in this experiment."
    )

id_to_idx = pd.Series(
    np.arange(len(node_df)),
    index=node_df["txId"]
)

node_features_raw = (
    node_df[FEATURE_COLS]
    .replace([np.inf, -np.inf], np.nan)
)


# ================================================================
# 4. BUILD DIRECTED GRAPH
# ================================================================

src = id_to_idx.loc[
    edges_raw["txId1"].astype(str)
].to_numpy(dtype=np.int64)

dst = id_to_idx.loc[
    edges_raw["txId2"].astype(str)
].to_numpy(dtype=np.int64)

edge_forward = np.vstack([
    src,
    dst
])

edge_reverse = np.vstack([
    dst,
    src
])

edge_undirected = np.hstack([
    edge_forward,
    edge_reverse
])

print("\nGraph edges")
print("Original directed:", edge_forward.shape[1])
print("Symmetrized:", edge_undirected.shape[1])


# ================================================================
# 5. MAP LABELED NODES
# ================================================================

# Reconstruct 'ids_all' and 'ids_test' safely since they might be missing from variables
if 'ids_all' not in globals() and 'labelled' in globals():
    ids_all = labelled['txId'].to_numpy()

if 'ids_test' not in globals() and 'test_raw' in globals():
    ids_test = test_raw['txId'].to_numpy()

train_ids = pd.Index(
    pd.Series(ids_all).astype(str)
)

test_ids = pd.Index(
    pd.Series(ids_test).astype(str)
)

assert train_ids.is_unique
assert test_ids.is_unique

train_graph_idx = id_to_idx.loc[
    train_ids
].to_numpy(dtype=np.int64)

test_graph_idx = id_to_idx.loc[
    test_ids
].to_numpy(dtype=np.int64)

if 'y_all' not in globals() and 'labelled' in globals():
    y_all = labelled['label'].astype(int).to_numpy()

if 't_all' not in globals() and 'labelled' in globals():
    t_all = labelled['time_step'].astype(int).to_numpy()

y = np.asarray(y_all, dtype=int)
t = np.asarray(t_all, dtype=int)

assert len(y) == len(train_graph_idx)
assert len(t) == len(train_graph_idx)

assert np.array_equal(
    node_times[train_graph_idx].astype(int),
    t
)

assert has_observed_time[train_graph_idx].all()
assert has_observed_time[test_graph_idx].all()


# ================================================================
# 6. TEMPORAL GRAPH PREPROCESSING
# ================================================================

def prepare_graph(max_time, scaler_fit_end):

    """
    Temporal graph:
      - Nodes visible up to max_time
      - Edges retained only when both endpoints visible
      - Imputer/scaler fitted on historical nodes only
      - Unknown-time nodes excluded

    This uses a full-node tensor with masked edges.
    """

    fit_mask = (
        has_observed_time &
        (node_times <= scaler_fit_end)
    )

    visible_mask = (
        has_observed_time &
        (node_times <= max_time)
    )

    assert fit_mask.sum() > 0

    imputer = SimpleImputer(
        strategy="median",
        keep_empty_features=True
    )

    imputer.fit(
        node_features_raw.loc[fit_mask]
    )

    X = imputer.transform(
        node_features_raw
    )

    scaler = StandardScaler()

    scaler.fit(
        X[fit_mask]
    )

    X = scaler.transform(X)

    X = np.nan_to_num(
        X,
        nan=0,
        posinf=0,
        neginf=0
    )

    X = np.clip(X, -10, 10)

    # Nonvisible nodes do not contribute to message passing.
    X[~visible_mask] = 0.0

    edge_mask = (
        visible_mask[src] &
        visible_mask[dst]
    )

    selected_src = src[edge_mask]
    selected_dst = dst[edge_mask]

    forward = np.vstack([
        selected_src,
        selected_dst
    ])

    reverse = np.vstack([
        selected_dst,
        selected_src
    ])

    undirected = np.hstack([
        forward,
        reverse
    ])

    graph = {
        "x": torch.tensor(
            X,
            dtype=torch.float32,
            device=DEVICE
        ),

        "forward": torch.tensor(
            forward,
            dtype=torch.long,
            device=DEVICE
        ),

        "reverse": torch.tensor(
            reverse,
            dtype=torch.long,
            device=DEVICE
        ),

        "undirected": torch.tensor(
            undirected,
            dtype=torch.long,
            device=DEVICE
        ),

        "visible": visible_mask
    }

    return graph


# ================================================================
# 7. VANILLA GRAPHSAGE
# ================================================================

class VanillaGraphSAGE(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=128,
        embed_dim=64,
        dropout=0.3
    ):

        super().__init__()

        self.conv1 = SAGEConv(
            input_dim,
            hidden_dim
        )

        self.conv2 = SAGEConv(
            hidden_dim,
            embed_dim
        )

        self.classifier = nn.Linear(
            embed_dim,
            1
        )

        self.dropout = dropout

    def forward(self, graph):

        x = graph["x"]
        edge_index = graph["undirected"]

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(
            x,
            edge_index
        )

        x = F.relu(x)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        return self.classifier(x).squeeze(-1)


# ================================================================
# 8. DIRECTED GRAPHSAGE
# ================================================================

class DirectedSAGELayer(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim
    ):

        super().__init__()

        # For edge u -> v:
        # forward aggregates upstream nodes into v.
        self.in_conv = SAGEConv(
            input_dim,
            output_dim,
            root_weight=False
        )

        # Reverse edges aggregate downstream nodes into v.
        self.out_conv = SAGEConv(
            input_dim,
            output_dim,
            root_weight=False
        )

        self.self_linear = nn.Linear(
            input_dim,
            output_dim
        )

        self.combine = nn.Linear(
            output_dim * 3,
            output_dim
        )

    def forward(
        self,
        x,
        edge_forward,
        edge_reverse
    ):

        h_self = self.self_linear(x)

        h_in = self.in_conv(
            x,
            edge_forward
        )

        h_out = self.out_conv(
            x,
            edge_reverse
        )

        h = torch.cat(
            [
                h_self,
                h_in,
                h_out
            ],
            dim=-1
        )

        return self.combine(h)


class DirectedGraphSAGE(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=128,
        embed_dim=64,
        dropout=0.3
    ):

        super().__init__()

        self.layer1 = DirectedSAGELayer(
            input_dim,
            hidden_dim
        )

        self.layer2 = DirectedSAGELayer(
            hidden_dim,
            embed_dim
        )

        self.classifier = nn.Linear(
            embed_dim,
            1
        )

        self.dropout = dropout

    def forward(self, graph):

        x = graph["x"]

        forward = graph["forward"]
        reverse = graph["reverse"]

        x = self.layer1(
            x,
            forward,
            reverse
        )

        x = F.relu(x)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.layer2(
            x,
            forward,
            reverse
        )

        x = F.relu(x)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        return self.classifier(x).squeeze(-1)


# ================================================================
# 9. TRAINING UTILITIES
# ================================================================

def make_model(model_type):

    if model_type == "Vanilla":

        return VanillaGraphSAGE(
            input_dim=len(FEATURE_COLS),
            hidden_dim=HIDDEN_DIM,
            embed_dim=EMBED_DIM,
            dropout=DROPOUT
        ).to(DEVICE)

    if model_type == "Directed":

        return DirectedGraphSAGE(
            input_dim=len(FEATURE_COLS),
            hidden_dim=HIDDEN_DIM,
            embed_dim=EMBED_DIM,
            dropout=DROPOUT
        ).to(DEVICE)

    raise ValueError(model_type)


def get_training_components(train_end):

    mask = t <= train_end

    idx = torch.tensor(
        train_graph_idx[mask],
        dtype=torch.long,
        device=DEVICE
    )

    target = torch.tensor(
        y[mask],
        dtype=torch.float32,
        device=DEVICE
    )

    times = t[mask]

    sample_weights = np.exp(
        -DECAY * (train_end - times)
    )

    sample_weights /= sample_weights.mean()

    sample_weights = torch.tensor(
        sample_weights,
        dtype=torch.float32,
        device=DEVICE
    )

    n_pos = max(1, int(target.sum().item()))
    n_neg = max(1, len(target) - n_pos)

    pos_weight = torch.tensor(
        n_neg / n_pos,
        dtype=torch.float32,
        device=DEVICE
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight,
        reduction="none"
    )

    return idx, target, sample_weights, criterion


def train_epochs(
    model,
    graph,
    train_end,
    epochs
):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY)

    idx, target, weights, criterion = (
        get_training_components(train_end)
    )

    for epoch in range(1, epochs + 1):

        model.train()

        optimizer.zero_grad()

        logits = model(graph)

        loss_values = criterion(
            logits[idx],
            target
        )

        loss = (
            loss_values * weights
        ).mean()

        loss.backward()
        optimizer.step()

    return model


def predict_nodes(
    model,
    graph,
    node_indices
):

    model.eval()

    with torch.no_grad():

        logits = model(graph)

        pred = torch.sigmoid(
            logits[node_indices]
        ).cpu().numpy()

    return pred


# ================================================================
# 10. INTERNAL EPOCH SELECTION
# ================================================================

def select_best_epoch(
    model_type,
    inner_train_end,
    inner_valid_start,
    inner_valid_end,
    seed
):

    set_seed(seed)

    # Training graph: no future nodes.
    train_graph = prepare_graph(
        max_time=inner_train_end,
        scaler_fit_end=inner_train_end
    )

    # Internal validation graph:
    # visible through internal validation end.
    valid_graph = prepare_graph(
        max_time=inner_valid_end,
        scaler_fit_end=inner_train_end
    )

    model = make_model(model_type)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    idx, target, weights, criterion = (
        get_training_components(inner_train_end)
    )

    val_mask = (
        (t >= inner_valid_start) &
        (t <= inner_valid_end)
    )

    val_nodes = torch.tensor(
        train_graph_idx[val_mask],
        dtype=torch.long,
        device=DEVICE
    )

    val_labels = y[val_mask]

    assert len(np.unique(val_labels)) == 2, (
        "Internal validation must contain both classes."
    )

    best_auc = -np.inf
    best_epoch = MIN_EPOCHS

    patience_counter = 0

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()
        optimizer.zero_grad()

        logits = model(train_graph)

        losses = criterion(
            logits[idx],
            target
        )

        loss = (
            losses * weights
        ).mean()

        loss.backward()
        optimizer.step()

        model.eval()

        with torch.no_grad():

            val_logits = model(
                valid_graph
            )[val_nodes]

            val_pred = torch.sigmoid(
                val_logits
            ).cpu().numpy()

        val_auc = roc_auc_score(
            val_labels,
            val_pred
        )

        if val_auc > best_auc + 1e-5:

            best_auc = val_auc
            best_epoch = epoch

            patience_counter = 0

        else:

            patience_counter += 1

        if (
            epoch >= MIN_EPOCHS
            and patience_counter >= PATIENCE
        ):
            break

    print(
        f"  {model_type:<9} "
        f"Seed={seed} "
        f"BestEpoch={best_epoch} "
        f"InnerAUC={best_auc:.6f}"
    )

    del model, optimizer, train_graph, valid_graph
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_epoch, best_auc


# ================================================================
# 11. REFIT + EXTERNAL PREDICTION
# ================================================================

def fit_and_predict(
    model_type,
    train_end,
    predict_end,
    prediction_nodes,
    epochs,
    seed
):

    set_seed(seed)

    # Historical training graph.
    train_graph = prepare_graph(
        max_time=train_end,
        scaler_fit_end=train_end
    )

    # Prediction graph includes observed future nodes.
    pred_graph = prepare_graph(
        max_time=predict_end,
        scaler_fit_end=train_end
    )

    model = make_model(model_type)

    model = train_epochs(
        model=model,
        graph=train_graph,
        train_end=train_end,
        epochs=epochs
    )

    pred = predict_nodes(
        model,
        pred_graph,
        prediction_nodes
    )

    del model, train_graph, pred_graph
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pred


# ================================================================
# 12. LIGHTGBM BASELINE
# ================================================================

# Safe fallback build of FULL_COLS and GRAPH_COLS if not globally defined
if 'FULL_COLS' not in globals():
    if 'best_cols' in globals() and 'two_hop_cols' in globals():
        FULL_COLS = list(dict.fromkeys(best_cols + two_hop_cols))
    elif 'X_tr' in globals():
        FULL_COLS = list(X_tr.columns)
    else:
        FULL_COLS = list(dict.fromkeys(best_cols))

if 'GRAPH_COLS' not in globals():
    if 'advanced_cols' in globals():
        GRAPH_COLS = [c for c in advanced_cols if c in X_s48_train.columns]
    else:
        GRAPH_COLS = [c for c in ['in_degree', 'out_degree', 'total_degree', 'pagerank', 'hub_score', 'authority_score', 'clustering_coeff', 'component_size'] if c in X_s48_train.columns]

LGB_COLS = list(dict.fromkeys(
    list(FULL_COLS) + list(GRAPH_COLS)
))

def train_lgbm(
    train_end,
    valid_start=None,
    valid_end=None,
    final=False
):

    model = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.08,
        num_leaves=31,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    )

    if final:

        tr_mask = np.ones(
            len(y),
            dtype=bool
        )

        pred_X = X_s48_test[LGB_COLS]

    else:

        tr_mask = t <= train_end

        val_mask = (
            (t >= valid_start) &
            (t <= valid_end)
        )

        pred_X = X_s48_train.loc[
            val_mask,
            LGB_COLS
        ]

    weights = np.exp(
        -DECAY * (train_end - t[tr_mask])
    )

    weights /= weights.mean()

    model.fit(
        X_s48_train.loc[
            tr_mask,
            LGB_COLS
        ],
        y[tr_mask],
        sample_weight=weights
    )

    pred = model.predict_proba(
        pred_X
    )[:, 1]

    del model

    return pred


# ================================================================
# 13. ENSEMBLE FUNCTIONS
# ================================================================

def rank_transform(pred):

    pred = np.asarray(pred)

    return rankdata(
        pred,
        method="average"
    ) / len(pred)


def make_ensembles(
    lgb,
    vanilla,
    directed
):

    lgb_rank = rank_transform(lgb)
    vanilla_rank = rank_transform(vanilla)
    directed_rank = rank_transform(directed)

    return {
        "LightGBM": lgb,

        "VanillaGraphSAGE": vanilla,

        "DirectedGraphSAGE": directed,

        "Rank_Directed_90_10":
            0.9 * lgb_rank
            + 0.1 * directed_rank,

        "Rank_Directed_80_20":
            0.8 * lgb_rank
            + 0.2 * directed_rank,

        "Rank_Directed_75_25":
            0.75 * lgb_rank
            + 0.25 * directed_rank,

        "Rank_Vanilla_80_20":
            0.8 * lgb_rank
            + 0.2 * vanilla_rank,

        "Rank_All_80_10_10":
            0.8 * lgb_rank
            + 0.1 * vanilla_rank
            + 0.1 * directed_rank
    }


# ================================================================
# 14. TEMPORAL VALIDATION
# ================================================================

WINDOWS = [
    {
        "name": "W1_15_28",
        "inner_train_end": 10,
        "inner_valid_start": 11,
        "inner_valid_end": 14,
        "train_end": 14,
        "valid_start": 15,
        "valid_end": 28
    },

    {
        "name": "W2_22_35",
        "inner_train_end": 17,
        "inner_valid_start": 18,
        "inner_valid_end": 21,
        "train_end": 21,
        "valid_start": 22,
        "valid_end": 35
    },

    {
        "name": "W3_Gap_22_35",
        "inner_train_end": 10,
        "inner_valid_start": 11,
        "inner_valid_end": 14,
        "train_end": 14,
        "valid_start": 22,
        "valid_end": 35
    }
]

validation_records = []
epoch_records = []
correlation_records = []

# Cache selected epochs:
# W1 and W3 share the same training cutoff.
epoch_cache = {}


for window in WINDOWS:

    name = window["name"]

    print("\n" + "=" * 75)
    print("VALIDATION:", name)
    print("=" * 75)

    train_end = window["train_end"]
    valid_start = window["valid_start"]
    valid_end = window["valid_end"]

    val_mask = (
        (t >= valid_start) &
        (t <= valid_end)
    )

    val_labels = y[val_mask]

    val_nodes = torch.tensor(
        train_graph_idx[val_mask],
        dtype=torch.long,
        device=DEVICE
    )

    # --------------------------------------------------------
    # LightGBM
    # --------------------------------------------------------

    lgb_pred = train_lgbm(
        train_end=train_end,
        valid_start=valid_start,
        valid_end=valid_end
    )

    gnn_predictions = {}

    # --------------------------------------------------------
    # GNN Models
    # --------------------------------------------------------

    for model_type in ["Vanilla", "Directed"]:

        seed_predictions = []

        for seed_offset in range(N_SEEDS):

            current_seed = SEED + seed_offset

            cache_key = (
                model_type,
                train_end,
                current_seed
            )

            if cache_key not in epoch_cache:

                best_epoch, inner_auc = (
                    select_best_epoch(
                        model_type=model_type,
                        inner_train_end=window[
                            "inner_train_end"
                        ],
                        inner_valid_start=window[
                            "inner_valid_start"
                        ],
                        inner_valid_end=window[
                            "inner_valid_end"
                        ],
                        seed=current_seed
                    )
                )

                epoch_cache[cache_key] = (
                    best_epoch,
                    inner_auc
                )

                epoch_records.append({
                    "Model": model_type,
                    "Train_End": train_end,
                    "Seed": current_seed,
                    "Best_Epoch": best_epoch,
                    "Inner_AUC": inner_auc
                })

            best_epoch, inner_auc = (
                epoch_cache[cache_key]
            )

            pred = fit_and_predict(
                model_type=model_type,
                train_end=train_end,
                predict_end=valid_end,
                prediction_nodes=val_nodes,
                epochs=best_epoch,
                seed=current_seed
            )

            seed_predictions.append(pred)

        gnn_predictions[model_type] = np.mean(
            seed_predictions,
            axis=0
        )

    vanilla_pred = gnn_predictions["Vanilla"]
    directed_pred = gnn_predictions["Directed"]

    # --------------------------------------------------------
    # Ensemble
    # --------------------------------------------------------

    ensemble_predictions = make_ensembles(
        lgb=lgb_pred,
        vanilla=vanilla_pred,
        directed=directed_pred
    )

    for model_name, pred in ensemble_predictions.items():

        auc = roc_auc_score(
            val_labels,
            pred
        )

        pr_auc = average_precision_score(
            val_labels,
            pred
        )

        validation_records.append({
            "Window": name,
            "Model": model_name,
            "ROC_AUC": auc,
            "PR_AUC": pr_auc
        })

        print(
            f"{model_name:<25} "
            f"AUC={auc:.6f} "
            f"PR={pr_auc:.6f}"
        )

    # --------------------------------------------------------
    # Prediction Correlation
    # --------------------------------------------------------

    prediction_dict = {
        "LightGBM": lgb_pred,
        "Vanilla": vanilla_pred,
        "Directed": directed_pred
    }

    for a, b in [
        ("LightGBM", "Vanilla"),
        ("LightGBM", "Directed"),
        ("Vanilla", "Directed")
    ]:

        corr = spearmanr(
            prediction_dict[a],
            prediction_dict[b]
        ).statistic

        correlation_records.append({
            "Window": name,
            "Model_A": a,
            "Model_B": b,
            "Spearman": corr
        })


# ================================================================
# 15. VALIDATION SUMMARY
# ================================================================

section52b_validation = pd.DataFrame(
    validation_records
)

section52b_summary = (
    section52b_validation
    .groupby("Model")
    .agg(
        Mean_AUC=("ROC_AUC", "mean"),
        Min_AUC=("ROC_AUC", "min"),
        Mean_PR=("PR_AUC", "mean")
    )
    .sort_values(
        "Mean_AUC",
        ascending=False
    )
)

section52b_epochs = pd.DataFrame(
    epoch_records
)

section52b_correlations = pd.DataFrame(
    correlation_records
)

print("\n" + "=" * 75)
print("SECTION 52B — VALIDATION SUMMARY")
print("=" * 75)

display(
    section52b_validation.pivot(
        index="Window",
        columns="Model",
        values="ROC_AUC"
    ).round(6)
)

display(
    section52b_summary.round(6)
)

print("\nSelected Epochs")

display(
    section52b_epochs
)

print("\nPrediction Correlations")

display(
    section52b_correlations.round(5)
)


# ================================================================
# 16. FINAL EPOCH SELECTION
# ================================================================

print("\n" + "=" * 75)
print("FINAL EPOCH SELECTION")
print("=" * 75)

FINAL_TRAIN_END = int(t.max())

FINAL_PREDICT_END = int(
    np.nanmax(node_times)
)

# Select epochs using an internal historical split.
# Then refit on all labeled training nodes through t=35.

FINAL_INNER_TRAIN_END = 28
FINAL_INNER_VALID_START = 29
FINAL_INNER_VALID_END = 35

final_gnn_predictions = {}

test_nodes = torch.tensor(
    test_graph_idx,
    dtype=torch.long,
    device=DEVICE
)

for model_type in ["Vanilla", "Directed"]:

    seed_predictions = []

    for seed_offset in range(N_SEEDS):

         current_seed = SEED + seed_offset

         best_epoch, inner_auc = select_best_epoch(
             model_type=model_type,
             inner_train_end=FINAL_INNER_TRAIN_END,
             inner_valid_start=FINAL_INNER_VALID_START,
             inner_valid_end=FINAL_INNER_VALID_END,
             seed=current_seed
         )

         print(
             f"Final {model_type}: "
             f"Seed={current_seed}, "
             f"Epochs={best_epoch}"
         )

         pred = fit_and_predict(
             model_type=model_type,
             train_end=FINAL_TRAIN_END,
             predict_end=FINAL_PREDICT_END,
             prediction_nodes=test_nodes,
             epochs=best_epoch,
             seed=current_seed
         )

         seed_predictions.append(pred)

    final_gnn_predictions[model_type] = np.mean(
        seed_predictions,
        axis=0
    )


# ================================================================
# 17. FINAL LIGHTGBM
# ================================================================

print("\nTraining final LightGBM...")

lgb_test_pred = train_lgbm(
    train_end=FINAL_TRAIN_END,
    final=True
)

final_ensembles = make_ensembles(
    lgb=lgb_test_pred,
    vanilla=final_gnn_predictions["Vanilla"],
    directed=final_gnn_predictions["Directed"]
)


# ================================================================
# 18. SUBMISSION ALIGNMENT
# ================================================================

prediction_ids = pd.Index(
    pd.Series(ids_test).astype(str)
)

original_test_ids = pd.Index(
    test_raw["txId"].astype(str)
)

assert prediction_ids.is_unique
assert original_test_ids.is_unique

positions = prediction_ids.get_indexer(
    original_test_ids
)

assert (positions >= 0).all()

assert len(sample) == len(test_raw)

assert sample.columns.tolist() == [
    "index",
    "target"
]


def save_submission(pred, filename):

    pred = np.asarray(pred)

    assert len(pred) == len(prediction_ids)
    assert np.isfinite(pred).all()

    submission = sample.copy()

    submission["target"] = pred[positions]

    assert submission["index"].equals(
        sample["index"]
    )

    path = OUTPUT_DIR / filename

    submission.to_csv(
         path,
         index=False
    )

    print("Saved:", path)


# ================================================================
# 19. SAVE SUBMISSIONS
# ================================================================

submission_map = {
    "DirectedGraphSAGE":
        "submission_s52b_directed.csv",

    "Rank_Directed_90_10":
        "submission_s52b_rank_90_10.csv",

    "Rank_Directed_80_20":
        "submission_s52b_rank_80_20.csv",

    "Rank_Directed_75_25":
         "submission_s52b_rank_75_25.csv",

    "Rank_All_80_10_10":
         "submission_s52b_rank_all.csv"
}

for name, filename in submission_map.items():

    save_submission(
         final_ensembles[name],
         filename
    )


# ================================================================
# 20. SAVE EXPERIMENT RESULTS
# ================================================================

section52b_validation.to_csv(
    OUTPUT_DIR / "section52b_validation.csv",
    index=False
)

section52b_summary.to_csv(
    OUTPUT_DIR / "section52b_summary.csv"
)

section52b_epochs.to_csv(
    OUTPUT_DIR / "section52b_epoch_selection.csv",
    index=False
)

section52b_correlations.to_csv(
    OUTPUT_DIR / "section52b_correlations.csv",
    index=False
)

print("\n" + "=" * 75)
print("SECTION 52B COMPLETE")
print("=" * 75)